<img src="https://storage.googleapis.com/kaggle-datasets-images/681625/1196732/a1d8dfe666ca94f660058a2f56a5639d/dataset-cover.png?t=2020-05-29-09-01-54">

# 🗾 Aerial Image for Semantic Segmentation

# Getting started
## Installing packages

In [1]:
!pip install tensorflow[and-cuda]==2.14
!pip install image-classifiers
!pip install gdown
!pip install PyMuPDF

from IPython.display import clear_output
clear_output(wait=False)

In [ ]:
!wget -O /kaggle/working/PlotNeuralNet.zip https://github.com/aletbm/Aerial_Image_Segmentation/raw/refs/heads/main/PlotNeuralNet.zip
!unzip -o PlotNeuralNet.zip
clear_output(wait=False)

## Importing packages

In [ ]:
import numpy as np
import pandas as pd
import os
import random

import cv2
from PIL import ImageColor

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.backend import epsilon
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, LearningRateScheduler, Callback, CSVLogger
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.regularizers import L2
from tensorflow.keras.losses import BinaryFocalCrossentropy, CategoricalFocalCrossentropy
from tensorflow.keras.utils import plot_model
from tensorflow.keras.metrics import BinaryCrossentropy, CategoricalAccuracy, OneHotIoU
from tensorflow.image import (resize_with_pad,
                              resize,
                              crop_to_bounding_box,
                              image_gradients,
                              random_brightness,
                              random_contrast,
                              random_flip_left_right as random_horizontal_flip,
                              random_flip_up_down as random_vertical_flip,
                              random_hue,
                              random_jpeg_quality,
                              random_saturation)

import keras
from keras import Model, Input
from keras.layers import (Layer,
                          Conv2D,
                          MaxPooling2D,
                          UpSampling2D,
                          Conv2DTranspose,
                          BatchNormalization,
                          ReLU,
                          SpatialDropout2D,
                          Dropout,
                          Add,
                          Concatenate)

from classification_models.keras import Classifiers

from IPython.display import clear_output
import fitz
import PIL
import gdown

plt.style.use("dark_background")

## Looking for available GPUs and what version of Tensorflow we are working on

In [ ]:
device_name = tf.test.gpu_device_name()
if "GPU" not in device_name:
    print("GPU device not found")
print(f"Found GPU at: {format(device_name)}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")
print(f"Tensorflow version: {tf.__version__}")

## Setting for reproducibility

In [ ]:
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)

## Gathering and defining file paths

In [ ]:
source_path = "/kaggle/input/"
dst_path = "/kaggle/working/"

img_paths = [#source_path + "modified-uavid-dataset/modified_uavid_dataset/train_data/Images",
             #source_path + "modified-uavid-dataset/modified_uavid_dataset/val_data/Images",
             #source_path + "semantic-segmentation-drone-dataset/classes_dataset/classes_dataset/original_images",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 1/images",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 2/images",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 3/images",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 4/images",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 5/images",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 6/images",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 7/images",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 8/images",
             #source_path + "satellite-image-and-mask/train_image",
             #source_path + "gaofen-satellite-images-five-billion-pixels/Image__8bit_NirRGB",
             source_path + "urban-segmentation-isprs/Potsdam/Images",
             source_path + "urban-segmentation-isprs/Vaihingen/Images",
             source_path + "swiss-drone-and-okutama-drone-datasets/images/train",
             source_path + "swiss-drone-and-okutama-drone-datasets/images/test",
             source_path + "swiss-drone-and-okutama-drone-datasets/images/val",
             source_path + "global-land-cover-mapping-openearthmap/images/train",
             source_path + "global-land-cover-mapping-openearthmap/images/val"
            ]

msk_paths = [#source_path + "modified-uavid-dataset/modified_uavid_dataset/train_data/Labels",
             #source_path + "modified-uavid-dataset/modified_uavid_dataset/val_data/Labels",
             #source_path + "semantic-segmentation-drone-dataset/classes_dataset/classes_dataset/label_images_semantic",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 1/masks",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 2/masks",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 3/masks",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 4/masks",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 5/masks",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 6/masks",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 7/masks",
             source_path + "semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 8/masks",
             #source_path + "satellite-image-and-mask/train_mask",
             #source_path + "gaofen-satellite-images-five-billion-pixels/Annotation__color",
             source_path + "urban-segmentation-isprs/Potsdam/Labels",
             source_path + "urban-segmentation-isprs/Vaihingen/Labels",
             source_path + "swiss-drone-and-okutama-drone-datasets/ground_truth/train",
             source_path + "swiss-drone-and-okutama-drone-datasets/ground_truth/test",
             source_path + "swiss-drone-and-okutama-drone-datasets/ground_truth/val",
             source_path + "global-land-cover-mapping-openearthmap/label/train",
             source_path + "global-land-cover-mapping-openearthmap/label/val"
            ]

dt_abb = [#"MUD", "MUD",
          #"SSDD",
          "SSAI", "SSAI", "SSAI", "SSAI", "SSAI", "SSAI", "SSAI", "SSAI",
          #"SIM",
          #"FBP",
          "ISPRS", "ISPRS",
          "SO", "SO", "SO",
          "OEM", "OEM"
         ]

### Creating a CSV file to join image and mask file paths

In [ ]:
if not os.path.exists(dst_path+"df_files.csv"):
    df_files = pd.DataFrame()
    imgs, msks = [], []
    for i, m, a in zip(img_paths, msk_paths, dt_abb):
        img_list = os.listdir(i)
        msk_list = os.listdir(m)
        img_list.sort()
        msk_list.sort()
        df_aux = pd.DataFrame(data={"filename_image": img_list, "filename_mask": msk_list})
        df_aux["filename_image"] = df_aux["filename_image"].apply(lambda x: i+'/'+x)
        df_aux["filename_mask"] = df_aux["filename_mask"].apply(lambda x: m+'/'+x)
        df_aux["dataset_abb"] = a
        df_files = pd.concat([df_files, df_aux], ignore_index=True)
    df_files.to_csv(dst_path+"df_files.csv", index=False)
else:
    df_files = pd.read_csv(dst_path+"df_files.csv")

df_files

# Exploratory and data analysis (EDA)

## Defining useful functions

### Functions for loading images and masks

In [ ]:
def load_img(filename):
    image = tf.convert_to_tensor(cv2.imread(filename)[..., ::-1])
    return image

def get_data(df, dataset=None):
    if dataset is None:
        num_images = len(df)
        random_idx = random.randint(0, num_images - 1)
        filename_img, filename_mask, dt_abb = df.loc[random_idx]
    else:
        df_aux = df_files[df_files["dataset_abb"] == dataset]
        num_images = len(df_aux)
        random_idx = random.randint(0, num_images - 1)
        filename_img, filename_mask, dt_abb = df_aux.iloc[random_idx]

    image = load_img(filename_img)
    mask = load_img(filename_mask)
    return image, mask, dt_abb

### Functions to normalize mask colors

In [ ]:
def norm_colors(mask, colors, colors_conv):
    colors = tf.cast(colors, dtype=tf.uint8)
    colors_conv = tf.cast(colors_conv, dtype=tf.uint8)
    n_colors = tf.size(colors)//3
    boolean_all = tf.TensorArray(dtype=tf.bool, size=n_colors)

    for i in tf.range(n_colors):
        x, y = colors[i], colors_conv[i]
        boolean_mask = tf.reduce_all(tf.equal(mask, x), axis=-1)
        boolean_all = boolean_all.write(i, boolean_mask)
        r, g, b = y[0], y[1], y[2]
        R = tf.where(boolean_mask, r, mask[..., 0])
        G = tf.where(boolean_mask, g, mask[..., 1])
        B = tf.where(boolean_mask, b, mask[..., 2])
        mask = tf.stack([R, G, B], axis=-1)

    boolean_all = tf.reduce_any(boolean_all.stack(), axis=0)
    unlabeled = tf.constant((0, 0, 0), dtype=tf.uint8)
    R = tf.where(boolean_all, mask[..., 0], unlabeled[0])
    G = tf.where(boolean_all, mask[..., 1], unlabeled[1])
    B = tf.where(boolean_all, mask[..., 2], unlabeled[2])
    mask = tf.stack([R, G, B], axis=-1)
    return mask

### Functions for resizing images and masks

In [ ]:
def resize_image(img=None, major_size=None, padding=True):
    if padding:
        image = resize_with_pad(img, method="nearest", target_height=major_size, target_width=major_size)
    else:
        img_shape = tf.shape(img)[0:2]
        ar = img_shape[1]/img_shape[0] #W/H
        image = resize(img, method="nearest", size=(major_size, major_size*ar) if img_shape[0] > img_shape[1] else (major_size/ar, major_size))
    return image

### Functions for plotting images and masks

In [ ]:
def get_patches(mask, list_colors, list_classes):
    colors, _ = tf.raw_ops.UniqueV2(x=tf.reshape(mask, (-1, 3)), axis=[0])
    colors = tf.cast(colors, dtype=tf.int32)
    patches = []
    color_idx = tf.expand_dims(tf.zeros(tf.shape(colors)[0], dtype=tf.bool), axis=-1)
    for i, color in enumerate(list_colors):
        color_idx = tf.concat([color_idx, tf.expand_dims(tf.reduce_all(colors == color, 1), axis=-1)], axis=-1)
    color_idx = tf.cast(tf.argmax(color_idx, axis=1), dtype=tf.int32).numpy()

    color_idx = color_idx[color_idx > 0]-1

    for idx_classes in color_idx:
        color = tuple(list_colors[idx_classes].numpy()/255)
        obj = list_classes[idx_classes]
        patches.append(mpatches.Patch(color=color, label=obj))
    return patches

def subplot(image, num_row, num_cols, idx, title, fontsize=8, patches=None, cmap=None, vmin=None, vmax=None):
    plt.subplot(num_row, num_cols, idx)
    plt.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax, interpolation='none')
    plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
    plt.title(title, fontsize=fontsize)
    if patches:
        plt.legend(handles=patches, fontsize=5.3, facecolor='white', labelcolor='black')
    return

def plot_sample(img, mask, df_colors, num_img=1, num_cols=2, ind=0, norm=False, color_correction=False, df_colors_conv=None, size_img=None):
    color_rgb = tf.constant(df_colors["Color_RGB"].tolist())
    classes = df_colors["Classes"].tolist()

    ind += 1
    subplot(image=img, num_row=num_img, num_cols=num_cols, idx=ind, title="Aerial image")

    ind += 1
    patches = get_patches(mask, color_rgb, classes)
    subplot(image=mask, num_row=num_img, num_cols=num_cols, idx=ind, title="Original mask", patches=patches)

    if norm == True or color_correction == True:
        color_conv = tf.constant(df_colors["Conversion"].tolist())
        baseline_classes = df_colors_conv.get_classes()
        baseline_color_rgb = tf.constant(df_colors_conv.get_colors())

    if norm == True:
        ind += 1
        mask_norm = norm_colors(mask, color_rgb, color_conv)
        patches = get_patches(mask_norm, baseline_color_rgb, baseline_classes)
        subplot(image=mask_norm, num_row=num_img, num_cols=num_cols, idx=ind, title="Re-Colorized mask", patches=patches)

    if color_correction == True:
        ind += 1
        mask_cor = mask_color_correction(mask, color_rgb)
        mask_norm = norm_colors(mask_cor, color_rgb, color_conv)
        patches = get_patches(mask_norm, baseline_color_rgb, baseline_classes)
        subplot(image=mask_norm, num_row=num_img, num_cols=num_cols, idx=ind, title="Re-Colorized mask with pixel color correction", patches=patches)
    return

def plot_samples(dataset, df_colors, df_files, num_img=1, w_disp=8, h_disp=5, norm=False, color_correction=False, df_colors_conv=None, size_img=None):
    num_cols = 2
    if norm == True:
        num_cols += 1
    if color_correction == True:
        num_cols += 1

    plt.subplots(num_img, num_cols, figsize=(w_disp, h_disp))

    for i in range(num_img):
        df_colors_ = df_colors.copy()
        ind = i*num_cols
        img, mask, abb = get_data(df_files, dataset=dataset)

        if size_img is not None:
            img = resize_image(img, size_img)
            mask = resize_image(mask, size_img)

        if abb == "SO" or abb == "OEM":
            color_rgb = tf.constant(df_colors_["Color_RGB"].tolist())
            re_color = tf.constant(df_colors_["Re-Color"].tolist())
            mask = norm_colors(mask, color_rgb, re_color)
            df_colors_["Color_RGB"] = df_colors_["Re-Color"]

        if abb == "ISPRS" or abb == "FBP":
            if abb == "FBP":
                img = img[..., ::-1]
            mask = mask[..., ::-1]

        plot_sample(img, mask, df_colors_, num_img, num_cols, ind, norm, color_correction, df_colors_conv)
    return

size = 512

### Class for handling labels

In [ ]:
class myLabels():
    def __init__(self):
        self.classes = {"Building":[60, 16, 152],
                        "Environment":[132, 41, 246],
                        "Road":[110, 193, 228],
                        "Unlabeled":[0, 0, 0]}

    def get_color(self, cat):
        return self.classes[cat]

    def get_colors(self):
        return list(self.classes.values())

    def get_class(self, clr):
        for class_, color in self.classes.items():
            if clr == color:
                break
        return class_

    def get_classes(self):
        return list(self.classes.keys())

Labels = myLabels()
num_classes = len(Labels.get_classes())

## Working on each dataset
### Dataset: Semantic segmentation of aerial image

In [ ]:
print(f'Number of images in the dataset: {df_files[df_files["dataset_abb"] == "SSAI"]["dataset_abb"].count()}')

#### Colors and classes in the masks

In [ ]:
seg_classes = ["Building",
               "Land",
               "Road",
               "Vegetation",
               "Water",
               "Unlabeled",
               "Unlabeled",
              ]

color_hex = ["#3C1098",
             "#8429F6",
             "#6EC1E4",
             "#FEDD3A",
             "#E2A929",
             "#000000",
             "#9B9B9B",
            ]

color_rgb = [np.array(ImageColor.getcolor(hex_, "RGB")) for hex_ in color_hex]
SSAI_colors = pd.DataFrame(data={"Classes": seg_classes, "Color_HEX": color_hex, "Color_RGB": color_rgb})
SSAI_colors

In [ ]:
plot_samples("SSAI", SSAI_colors, df_files, size_img=size)

#### Conversion of colors

In [ ]:
SSAI_colors["Conversion"] = [Labels.get_color("Building"),
                             Labels.get_color("Environment"),
                             Labels.get_color("Road"),
                             Labels.get_color("Environment"),
                             Labels.get_color("Environment"),
                             Labels.get_color("Unlabeled"),
                             Labels.get_color("Unlabeled")]

SSAI_colors = SSAI_colors.iloc[:-1]
SSAI_colors

In [ ]:
plot_samples("SSAI", SSAI_colors, df_files, num_img=1, norm=True, df_colors_conv=Labels, h_disp=5, w_disp=12, size_img=size)

### Correcting the color of mask pixels

While mask re-colorization process we wrongly introduce black halos due to pixel color weakness in frontier between two different classes when we resize image. We must correct this problem since this introduce a missclassification of pixels.

But, how to resolve this problem? Let us see RGB domain in a 3 dimensional-space:

<img src="https://i.ibb.co/SwrPMPF/RGB-3d.png" width=200>

The RGB domain can be represented by a cube, where each point inside cube represent a color, each point have 3 components that determine its position.
The distance between two points inside cube can be calculated with Euclidean distance equation for two points in 3 dimensional-space:

<img src="https://i.ibb.co/QdDQ40C/589px-Euclidean-distance-3d-2-cropped.png" width=300>

The distance is:

$d(p, q)=\sqrt{|p_1-q_1|^2 + |p_2-q_2|^2 + |p_3-q_3|^2} \quad\text{where} \quad p=(p_1,p_2,p_3),q=(q_1,q_2,q_3) \quad\text{any two points}$

This way we can approximate the weak colors to most near color class.

In [ ]:
def euclidean_distance(mask, color):
    sum_ = tf.cast(tf.reduce_sum(tf.math.pow(mask-color, 2), axis=2), dtype=tf.float32)
    return tf.math.sqrt(sum_)

def mask_color_correction(mask, colors):
    n_colors = tf.size(colors)//3
    mask_ = tf.cast(mask, dtype=tf.int32)
    mask_cor = tf.TensorArray(dtype=tf.float32, size=n_colors)

    for i in tf.range(n_colors):
        mask_cor = mask_cor.write(i, euclidean_distance(mask_, colors[i]))
    mask_cor = tf.transpose(mask_cor.stack(), perm=[1, 2, 0])

    mask_cor = tf.cast(tf.argmin(mask_cor, axis=2), dtype=tf.int32)
    mask_bool = tf.cast(tf.tile(tf.expand_dims(tf.clip_by_value(tf.reduce_max(mask, 2), 0, 1), -1), [1, 1, 3]), dtype=tf.uint8)
    mask_cor = tf.cast(tf.gather(colors, mask_cor), dtype=tf.uint8)
    return mask_bool * mask_cor

plt.subplots(1, 3, figsize=(12, 5))

img = load_img("/kaggle/input/semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 5/images/image_part_007.jpg")
mask = load_img("/kaggle/input/semantic-segmentation-of-aerial-imagery/Semantic segmentation dataset/Tile 5/masks/image_part_007.png")
colors_rgb = tf.constant(SSAI_colors["Color_RGB"].to_list())
colors_conv = tf.constant(SSAI_colors["Conversion"].to_list())

img = resize_image(img, size)
mask = resize_image(mask, size)

plt.subplot(1, 3, 1)
plt.imshow(mask)
plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
plt.title("Original mask", fontsize=8)

plt.subplot(1, 3, 2)
plt.imshow(norm_colors(mask, colors_rgb, colors_conv))
plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
plt.title("Re-colorized mask with black halos", fontsize=8)

plt.subplot(1, 3, 3)
mask_cor = mask_color_correction(mask, colors_rgb)
plt.imshow(norm_colors(mask_cor, colors_rgb, colors_conv))
plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
plt.title("Re-colorized mask with pixel color correction", fontsize=8);

This effect appreciate more in the next dataset.

### Dataset: Urban Segmentation - ISPRS

In [ ]:
print(f'Number of images in the dataset: {df_files[df_files["dataset_abb"] == "ISPRS"]["dataset_abb"].count()}')

#### Colors and classes in the masks

In [ ]:
seg_classes = ["Impervious surfaces",
               "Building",
               "Low vegetation",
               "Tree",
               "Car",
               "Clutter/background",
               "Background",]

color_bgr = [(255, 255, 255),
             (0, 0, 255),
             (0, 255, 255),
             (0, 255, 0),
             (255, 255, 0),
             (255, 0, 0),
             (0, 0, 0),]

color_bgr = np.array([np.array(color) for color in color_bgr])

color_rgb = np.stack([color_bgr[..., 2], color_bgr[..., 1], color_bgr[..., 0]], axis=-1)
color_rgb = [np.array(color) for color in color_rgb]

ISPRS_colors = pd.DataFrame(data={"Classes": seg_classes, "Color_RGB": color_rgb})
ISPRS_colors

In [ ]:
plot_samples("ISPRS", ISPRS_colors, df_files, size_img=size)

#### Conversion of colors
##### Conversion table

<table>
    <tr><th>ISPRS categories</th><th>Baseline</th></tr>
    <tr><td>Impervious surfaces</td><td>Road</td></tr>
    <tr><td>Building</td><td>Building</td></tr>
    <tr><td>Low vegetation</td><td>Environment</td></tr>
    <tr><td>Tree</td><td>Environment</td></tr>
    <tr><td>Car</td><td>Environment</td></tr>
    <tr><td>Clutter/background</td><td>Environment</td></tr>
    <tr><td>Background</td><td>Unlabeled</td></tr>
</table>

In [ ]:
ISPRS_colors["Conversion"] = [Labels.get_color("Road"),
                              Labels.get_color("Building"),
                              Labels.get_color("Environment"),
                              Labels.get_color("Environment"),
                              Labels.get_color("Environment"),
                              Labels.get_color("Environment"),
                              Labels.get_color("Unlabeled"),]

ISPRS_colors

In [ ]:
plot_samples("ISPRS", ISPRS_colors, df_files, w_disp=20, norm=True, color_correction=True, df_colors_conv=Labels, size_img=size)

### Dataset: Swiss Drone and Okutama Drone Datasets

In [ ]:
print(f'Number of images in the dataset: {df_files[df_files["dataset_abb"] == "SO"]["dataset_abb"].count()}')

#### Colors and classes in the masks

In [ ]:
seg_classes = ["People",
               "Water",
               "Wheeled vehicles",
               "Plants",
               "Train tracks",
               "Non-paved ground",
               "Paved ground",
               "Buildings",
               "Outdoor structures",
               "Background",
              ]

base = np.array([1, 1, 1])

color_rgb = [i*base for i in range(len(seg_classes))][::-1]

re_color_rgb = [[240, 255, 0],
                [0, 168, 255],
                [6, 0, 130],
                [31, 123, 22],
                [128, 0, 128],
                [189, 107, 0],
                [135, 135, 135],
                [181, 0, 0],
                [237, 237, 237],
                [0, 0, 0],
               ]

SO_colors = pd.DataFrame(data={"Classes": seg_classes, "Color_RGB": color_rgb, "Re-Color": re_color_rgb})
SO_colors

In [ ]:
plot_samples("SO", SO_colors, df_files, size_img=size)

#### Conversion of colors
##### Conversion table
<table>
    <tr><th>SO categories</th><th>Baseline</th></tr>
    <tr><td>People</td><td>Environment</td></tr>
    <tr><td>Water</td><td>Environment</td></tr>
    <tr><td>Wheeled vehicles</td><td>Environment</td></tr>
    <tr><td>Plants</td><td>Environment</td></tr>
    <tr><td>Train tracks</td><td>Road</td></tr>
    <tr><td>Non-paved ground</td><td>Environment</td></tr>
    <tr><td>Paved ground</td><td>Road</td></tr>
    <tr><td>Buildings</td><td>Building</td></tr>
    <tr><td>Outdoor structures</td><td>Environment</td></tr>
    <tr><td>Background</td><td>Unlabeled</td></tr>
</table>

In [ ]:
SO_colors["Conversion"] = [Labels.get_color("Environment"),
                           Labels.get_color("Environment"),
                           Labels.get_color("Environment"),
                           Labels.get_color("Environment"),
                           Labels.get_color("Road"),
                           Labels.get_color("Environment"),
                           Labels.get_color("Road"),
                           Labels.get_color("Building"),
                           Labels.get_color("Environment"),
                           Labels.get_color("Unlabeled"),
                          ]

SO_colors

In [ ]:
plot_samples("SO", SO_colors, df_files, w_disp=16, norm=True, color_correction=True, df_colors_conv=Labels, size_img=size)

### Dataset: Global Land Cover Mapping - OpenEarthMap

In [ ]:
print(f'Number of images in the dataset: {df_files[df_files["dataset_abb"] == "OEM"]["dataset_abb"].count()}')

#### Colors and classes in the masks

In [ ]:
seg_classes = ["Building",
               "Agriculture land",
               "Water",
               "Tree",
               "Road",
               "Developed space",
               "Rangeland",
               "Bareland",
               "Background",
              ]

color_hex = ["#DE1F07",
             "#4BB549",
             "#0045FF",
             "#226126",
             "#FFFFFF",
             "#949494",
             "#00FF24",
             "#800000",
             "#000000",
            ]

base = np.array([1, 1, 1])
color_rgb = [i*base for i in range(len(seg_classes))][::-1]

re_color_rgb = [np.array(ImageColor.getcolor(hex_, "RGB")) for hex_ in color_hex]

OEM_colors = pd.DataFrame(data={"Classes": seg_classes, "Color_RGB": color_rgb, "Re-Color": re_color_rgb})
OEM_colors

In [ ]:
plot_samples("OEM", OEM_colors, df_files)

#### Conversion of colors
##### Conversion table

<table>
    <tr><th>OEM categories</th><th>Baseline</th></tr>
    <tr><td>Building</td><td>Building</td></tr>
    <tr><td>Agriculture land</td><td>Environment</td></tr>
    <tr><td>Water</td><td>Environment</td></tr>
    <tr><td>Tree</td><td>Environment</td></tr>
    <tr><td>Road</td><td>Road</td></tr>
    <tr><td>Developed space</td><td>Road</td></tr>
    <tr><td>Rangeland</td><td>Environment</td></tr>
    <tr><td>Bareland</td><td>Environment</td></tr>
    <tr><td>Background</td><td>Unlabeled</td></tr>
</table>

In [ ]:
OEM_colors["Conversion"] = [Labels.get_color("Building"),
                           Labels.get_color("Environment"),
                           Labels.get_color("Environment"),
                           Labels.get_color("Environment"),
                           Labels.get_color("Road"),
                           Labels.get_color("Road"),
                           Labels.get_color("Environment"),
                           Labels.get_color("Environment"),
                           Labels.get_color("Unlabeled")]

OEM_colors

In [ ]:
plot_samples("OEM", OEM_colors, df_files, w_disp=16, norm=True, color_correction=True, df_colors_conv=Labels, size_img=size)

# Creating patches from big image

## Defining useful functions

In [ ]:
def normalize_size(image, size_img):
    img_shape = tf.shape(image)[:2]
    min_side = tf.reduce_min(img_shape)
    max_side = tf.reduce_max(img_shape)
    ar = max_side/min_side

    if min_side != size_img:
        max_side_resize = size_img * ar
        image = resize_image(image, max_side_resize, padding=False)
    return image

def crop_image(image, size_img):
    img_shape = tf.shape(image)
    h = img_shape[0]
    w = img_shape[1]

    n_rows = tf.cast(tf.math.ceil(h/size_img), dtype=tf.int32)
    n_cols = tf.cast(tf.math.ceil(w/size_img), dtype=tf.int32)

    crops_image = tf.TensorArray(dtype=tf.uint8, size=n_rows*n_cols)

    i = 0
    for r in range(n_rows):
        h0 = h-size_img if h < size_img*(1 + r) else size_img*r
        for c in range(n_cols):
            w0 = w-size_img if w < size_img*(1 + c) else size_img*c
            crops_image = crops_image.write(i, crop_to_bounding_box(image, h0, w0, size_img, size_img))
            i += 1

    return crops_image.stack()

## Loading a example

In [ ]:
img = load_img("/kaggle/input/urban-segmentation-isprs/Potsdam/Images/top_potsdam_2_10_RGB.tif")
msk = load_img("/kaggle/input/urban-segmentation-isprs/Potsdam/Labels/top_potsdam_2_10_label.tif")[..., ::-1]

#img = load_img("/kaggle/input/swiss-drone-and-okutama-drone-datasets/images/train/okutama_02_50_016.png")
#msk = load_img("/kaggle/input/swiss-drone-and-okutama-drone-datasets/ground_truth/train/okutama_02_50_016.png")[..., ::-1]

#img = tf.image.rot90(img)
#msk = tf.image.rot90(msk)

color_rgb = tf.constant(ISPRS_colors["Color_RGB"].to_list(), dtype=tf.int32)
color_conv = tf.constant(ISPRS_colors["Conversion"].to_list(), dtype=tf.int32)
mask_cor = mask_color_correction(msk, color_rgb)
mask_norm = norm_colors(mask_cor, color_rgb, color_conv)

plt.subplots(1, 2, figsize=(8, 12))
plot_sample(img, mask_norm, SSAI_colors)

## Extracting and plotting patches

In [ ]:
size_img = 256
image = normalize_size(img, 3*size_img)
mask = normalize_size(mask_norm, 3*size_img)
crops_image = crop_image(image, size_img)
crops_mask = crop_image(mask, size_img)

idx = 0
n_img = 4
fig, axs = plt.subplots(3, 4, figsize=(8, 6))
gs = axs[2, 2].get_gridspec()

for ax_ in axs[:2, :2]:
    for ax in ax_:
        ax.remove()
fig.add_subplot(gs[:2, :2])
plt.imshow(image)

h, w = tf.shape(image)[:2]
for i in range(np.ceil(w/size_img).astype(np.int32)):
    plt.axvline(x=size_img*i, lw=0.5)
for i in range(np.ceil(h/size_img).astype(np.int32)):
    plt.axhline(y=size_img*i, lw=0.5)

xv, yv = np.meshgrid(np.arange(size_img//2, h, size_img), np.arange(size_img//2, w, size_img))
xy_center = tf.reshape(tf.concat([tf.expand_dims(xv, -1), tf.expand_dims(yv, -1)], axis=2), shape=(-1, 2))
crop_number = np.arange(1, len(crops_image)+1)
for i, xy in enumerate(xy_center):
    x0 = xy[0] - 20
    y0 = xy[1] + 20
    plt.text(x0, y0, crop_number[i], fontsize=15)
plt.title("Resized original image", fontsize=8)
plt.tick_params(axis='x', which='major', labelsize=8)
plt.tick_params(axis='y', which='major', labelsize=8)

for i in range(n_img):
    idx_i, idx_m = [(3, 4), (7, 8), (11, 12), (9, 10)][i]
    plt.subplot(3, 4, idx_i)
    plt.imshow(crops_image[i])
    plt.title(f"Image patche {crop_number[i]}", fontsize=8)
    plt.tick_params(axis='x', which='major', labelsize=8)
    plt.tick_params(axis='y', which='major', labelsize=8)

    plt.subplot(3, 4, idx_m)
    plt.imshow(crops_mask[i])
    plt.title(f"Mask patche {crop_number[i]}", fontsize=8)
    plt.tick_params(axis='x', which='major', labelsize=8)
    plt.tick_params(axis='y', which='major', labelsize=8)

fig.tight_layout()

In [ ]:
print(f"We extracted {len(crops_image)} {size_img}x{size_img} sub-images from a {h.numpy()}x{w.numpy()} image.")

## Restore image from patches

In [ ]:
def restore_image(crops_image, image_shape):
    h = image_shape[0]
    w = image_shape[1]

    crop_size = crops_image[1].shape[0]

    n_rows = tf.cast(tf.math.ceil(h/crop_size), dtype=tf.int32)
    n_cols = tf.cast(tf.math.ceil(w/crop_size), dtype=tf.int32)

    rows = []
    for i in range(n_rows):
        crops = [i for i in crops_image[(i*n_cols):(i*n_cols)+n_cols]]
        restored_row = tf.concat(crops, axis=1)
        if w/crop_size < tf.cast(n_cols, dtype=tf.float64):
            crops[-1] = crops[-1][:, - (w-restored_row.shape[1]):]
        rows.append(tf.concat(crops, axis=1))

    restored_image = tf.concat(rows, axis=0)

    if restored_image.shape[0] > h:
        rows[-1] = rows[-1][(restored_image.shape[0]-h):, :]

    restored_image = tf.concat(rows, axis=0)
    return restored_image

img_shape = tf.shape(image)
restored_image = restore_image(crops_image, img_shape)
plt.imshow(restored_image)

# Joining the datasets

In [ ]:
list_abb = list(set(dt_abb))
list_abb.sort()
list_abb

In [ ]:
palettes = dict(zip(list_abb, [#FBP_colors,
                               ISPRS_colors,
                               #MUD_colors,
                               OEM_colors,
                               #SIM_colors,
                               SO_colors,
                               SSAI_colors,
                               #SSDD_colors
                              ]))
#palettes = [MUD_colors, SSDD_colors, SSAI_colors, SIM_colors, FBP_colors, ISPRS_colors, SO_colors]

## Creating a CSV file with all shuffled file paths

In [ ]:
df_files_shuffled = df_files[df_files["dataset_abb"].isin(["MUD", "SSDD", "SSAI", "ISPRS", "SO", "OEM"])]
df_files_shuffled = df_files_shuffled.sample(frac=1, random_state=seed)
df_files_shuffled

## Creating a TFRecordDataset for containing all preprocessed images and mask

In [ ]:
def get_palettes(abb):
    df_colors = palettes[abb]
    color_rgb = df_colors["Color_RGB"].to_list()
    color_conv = df_colors["Conversion"].to_list()
    return tf.cast(color_rgb, dtype=tf.int32), tf.cast(color_conv, dtype=tf.int32)

def normalize_colors(image, mask, abb):
    if abb == "ISPRS" or abb == "FBP":
        if abb == "FBP":
            crop_image = image[..., ::-1]
        mask = mask[..., ::-1]
    color_rgb, color_conv = get_palettes(abb)
    mask_cor = mask_color_correction(mask, color_rgb)
    mask = norm_colors(mask_cor, color_rgb, color_conv)
    return tf.cast(image, dtype=tf.uint8), tf.cast(mask, dtype=tf.uint8)

def createTFRecord(dataset, palettes, size_img):
    #options = tf.io.TFRecordOptions(compression_type="GZIP", compression_level=9)
    writer = tf.io.TFRecordWriter(dst_path+"my_dataset.tfrecord",
                                  #options=options
                                 )
    cnt_written_img = 0
    for index, row in dataset.iterrows():
        image = load_img(row.filename_image)
        mask = load_img(row.filename_mask)

        image = normalize_size(image, 3*size_img)
        mask = normalize_size(mask, 3*size_img)
        image, mask = normalize_colors(image, mask, row.dataset_abb)
        crops_image = crop_image(image, size_img)
        crops_mask = crop_image(mask, size_img)

        for image, mask in zip(crops_image, crops_mask):
            image_data, mask_data = image.numpy().tobytes(), mask.numpy().tobytes()
            example = tf.train.Example(features=tf.train.Features(feature={
                'image': tf.train.Feature(bytes_list=tf.train.BytesList(value=[image_data])),
                'mask': tf.train.Feature(bytes_list=tf.train.BytesList(value=[mask_data])),
            }))

            writer.write(example.SerializeToString())
            cnt_written_img += 1
    return cnt_written_img

def parse(feature):
    features = tf.io.parse_single_example(
        feature,
        features={
        'image': tf.io.FixedLenFeature([], tf.string),
        'mask': tf.io.FixedLenFeature([], tf.string),
    })
    image = tf.reshape(tf.io.decode_raw(features['image'], out_type=tf.uint8), shape=(size_img, size_img, 3))
    mask = tf.reshape(tf.io.decode_raw(features['mask'], out_type=tf.uint8), shape=(size_img, size_img, 3))
    return image, mask

autotune = tf.data.AUTOTUNE
create_tensorflow_record = False

if create_tensorflow_record == True:
    cnt_written_img = createTFRecord(df_files_shuffled, palettes, size_img=size_img) #cnt_written_img #28884
    dataset = tf.data.TFRecordDataset(dst_path+"my_dataset.tfrecord")

## Loading and visualazing the contain of a TFRecordDataset

In [ ]:
dataset = tf.data.TFRecordDataset(dst_path+"my_dataset.tfrecord",
                                  #compression_type="GZIP"
                                 )
dataset = dataset.map(parse, num_parallel_calls=autotune)
dataset = dataset.apply(tf.data.experimental.assert_cardinality(28884))

In [ ]:
plt.subplots(1, 2, figsize=(8, 5))
for image, mask in dataset.shuffle(1000).take(1):
    plot_sample(image, mask, SSAI_colors)

# Decomposing of RGB mask to binary masks

In [ ]:
def rgb2ohe(mask, colors):
    for i, color in enumerate(colors):
        boolean_mask = tf.reduce_all(tf.equal(mask, tf.constant(color, dtype=tf.uint8)), axis=-1)
        mask_i = tf.where(boolean_mask, 1., 0.)
        mask_i = tf.expand_dims(mask_i, axis=-1)
        if i == 0:
            mask_ohe = mask_i
        else:
            mask_ohe = tf.concat([mask_ohe, mask_i], axis=2)
    return mask_ohe

def delete_empty_subplots(fig, axs):
    for ax_row in axs:
        for ax in ax_row:
            if ax.title.get_text() == "":
                fig.delaxes(ax)
    return fig, axs

def plot_decomposition(image, mask=None, mask_ohe=None, df_colors=SSAI_colors, num_classes=0, edge=None):
    fig, axs = plt.subplots(3, 3, figsize=(8, 8))

    if mask is not None:
        plot_sample(image, mask, df_colors, num_img=3, num_cols=3)
        ind = 3
    else:
        subplot(image=image, num_row=3, num_cols=3, idx=1, title="Aerial image")
        ind = 2

    one_per_mask = tf.reduce_sum(tf.reduce_sum(mask_ohe, axis=0), axis=0)
    for i in range(num_classes):
        labels = Labels.get_classes()
        if one_per_mask[i] > 0:
            subplot(image=mask_ohe[..., i], num_row=3, num_cols=3, idx=ind, title=labels[i] + " mask", cmap=plt.cm.gray, vmin=0, vmax=1)
            ind += 1

    if edge is not None:
        subplot(image=edge, num_row=3, num_cols=3, idx=ind, title="Image edges", cmap=plt.cm.gray, vmin=0, vmax=1)

    fig, axs = delete_empty_subplots(fig, axs)
    plt.tight_layout()
    return

for image, mask in dataset.shuffle(200).take(1):
    mask_ohe = rgb2ohe(mask, Labels.get_colors())
    plot_decomposition(tf.cast(image, dtype=tf.uint8), mask, mask_ohe, SSAI_colors, num_classes=num_classes)

## Recomposing of binary masks to RGB mask

In [ ]:
def ohe2rgb(mask_ohe, colors):
    mask_max = tf.math.argmax(mask_ohe, axis=-1)
    mask_rgb = tf.zeros(shape=mask_max.shape+(3), dtype=tf.int32)
    colors = tf.constant(colors)
    for i, color in enumerate(colors):
        indices = tf.where(mask_max == i)
        color_ = tf.ones([indices.shape[0], 3], dtype=tf.int32) * color
        mask_rgb = tf.tensor_scatter_nd_update(mask_rgb, indices, updates=color_)
    #mask_rgb = tf.reshape(tf.gather(colors, indices), shape=mask_ohe.shape+(3))
    return mask_rgb

mask_rgb = ohe2rgb(mask_ohe, Labels.get_colors())
plt.imshow(mask_rgb)

# Mask handling to edge detection

## Using binary masks for edge detection

In [ ]:
def get_edges(mask_ohe, rgb_format=False):
    image_edge = tf.zeros(shape=mask_ohe.shape[:2], dtype=tf.int32)
    num_classes = mask_ohe.shape[-1]
    for i in range(num_classes):
        dy, dx = image_gradients(tf.expand_dims(tf.expand_dims(mask_ohe[..., i], axis=-1), axis=0))
        image_edge += tf.cast(tf.squeeze(dy**2 + dx**2), dtype=tf.int32)
        image_edge = tf.where(image_edge != 0, 1, 0)
    image_edge = tf.expand_dims(image_edge, axis=-1)
    if rgb_format:
        image_edge = tf.ones(shape=[256, 256, 3], dtype=tf.uint8) * tf.cast(image_edge, tf.uint8) * 255
    return image_edge

def overlap_images(image1, image2, alpha=0.4):
    beta = ( 1.0 - alpha )
    overlap_images = cv2.addWeighted(image1, alpha, image2, beta, gamma=0.1)
    return overlap_images

image_edge = get_edges(mask_ohe, rgb_format=True)
overlap_images = overlap_images(image.numpy(), image_edge.numpy())

plt.imshow(overlap_images)
plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)

## Using RGB masks for edge detection

In [ ]:
def get_edges_from_mask(mask_rgb, rgb_format=True, lw=1):
    mask_rgb = tf.cast(mask_rgb, dtype=tf.int32)
    image_edge = tf.zeros(shape=[256, 256, 1], dtype=tf.uint8)
    for i in range(lw):
        if i == 0:
            dx, dy = image_gradients(tf.expand_dims(mask_rgb, axis=0))
        else:
            dx, dy = image_gradients(tf.expand_dims(image_edge_, axis=0))

        image_edge_ = tf.cast(tf.squeeze(dy**2 + dx**2), dtype=tf.int32)
        image_edge_ = tf.where(image_edge_ != 0, [255, 255, 255], [0, 0, 0])
        image_edge += tf.cast(image_edge_, dtype=tf.uint8)

    if not rgb_format:
        image_edge = tf.reduce_max(image_edge, axis=-1, keepdims=True)
        image_edge = tf.where(image_edge != 0, 1, 0)
    #dx, dy = image_gradients(tf.expand_dims(tf.cast(image_edge, tf.int32), axis=0))
    #image_edge_ = tf.cast(tf.squeeze(dy**2 + dx**2), dtype=tf.int32)
    #image_edge_ = tf.cast(tf.where(image_edge_ != 0, [255, 255, 255], [0, 0, 0]), tf.uint8)
    return image_edge

def overlap_images(image1, image2, alpha=0.4):
    beta = ( 1.0 - alpha )
    overlap_images = cv2.addWeighted(image1, alpha, image2, beta, gamma=0.1)
    return overlap_images

image_edge = get_edges_from_mask(mask, rgb_format=True, lw=1)
overlap_images = overlap_images(image.numpy(), image_edge.numpy())

plt.imshow(overlap_images)
plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)

## Minor class augmentation to improve edge prediction

In [ ]:
image_edge = get_edges_from_mask(mask, rgb_format=False, lw=3)
plt.imshow(image_edge, cmap=plt.cm.gray, vmin=0, vmax=1)
plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)

# Classes weight

## Counting pixels and images per classes

In [ ]:
def rgb2hex(color):
    return ('#%02x%02x%02x' % tuple(color)).upper()

if not os.path.exists(dst_path+"countPerClass.csv"):
    colors_hex = [rgb2hex(color) for color in Labels.get_colors()]
    countPerClass = pd.DataFrame(data={"classes": Labels.get_classes(), "colors": Labels.get_colors(), "colors_hex": colors_hex})
    countPerClass["pixels"] = 0
    countPerClass["images"] = 0
    edge_pixels = 0

    for image, mask in dataset:
        colors, _, counts = tf.raw_ops.UniqueWithCountsV2(x=tf.reshape(mask, (-1, 3)), axis=[0])
        for color, count in zip(colors, counts):
            color = list(color.numpy())
            countPerClass.loc[countPerClass["classes"] == Labels.get_class(color), "pixels"] += count.numpy()
        countPerClass.loc[countPerClass["classes"] == Labels.get_class(color), "images"] += 1
        edge_pixels += tf.reduce_sum(tf.cast(get_edges_from_mask(mask, lw=3)/255, tf.int32)).numpy()
    countPerClass.to_csv(dst_path+"countPerClass.csv", index=False)

    countPerEdge = pd.DataFrame(data={"total_pixels": [countPerClass.pixels.sum()], "edge_pixels": [edge_pixels]})
    countPerEdge.to_csv(dst_path+"countPerEdge.csv", index=False)

countPerClass = pd.read_csv(dst_path+"countPerClass.csv")
countPerEdge = pd.read_csv(dst_path+"countPerEdge.csv")

In [ ]:
countPerClass

In [ ]:
countPerEdge

## Visualizing the imbalance in classes
### Number of images per class

In [ ]:
plt.figure(figsize=(10, 3))

countPerClass = countPerClass.sort_values("images", ascending=False)
ax = sns.barplot(countPerClass, y="classes", x=countPerClass["images"]/countPerClass.images.sum(), palette=countPerClass["colors_hex"], edgecolor="gray")
ax.set_xlabel("% [Images/Total of images]")
ax.bar_label(ax.containers[0], fontsize=8, fmt='%0.3f%%')
ax.set_title("Number of images per class");

### Number of pixels per class

In [ ]:
plt.figure(figsize=(10, 3))

countPerClass = countPerClass.sort_values("pixels", ascending=False)
ax = sns.barplot(countPerClass, y="classes", x=countPerClass["pixels"]/countPerClass.pixels.sum(), palette=countPerClass["colors_hex"], edgecolor="gray")
ax.set_xlabel("% [Pixels/Total of pixels]")
ax.bar_label(ax.containers[0], fontsize=8, fmt='%0.3f%%')
ax.set_title("Number of pixels per class");

## Calculating classes weigth

### Classes weigth for semantic segmentation

In [ ]:
classes = Labels.get_classes()
num_classes = len(classes)
countPerClass = countPerClass.sort_index()
#n_samples / (n_classes * np.bincount(y))
classes_weight = countPerClass.pixels[0:-1].sum() / (num_classes * countPerClass.pixels.values[0:-1])
classes_weight = np.append(classes_weight, [0])

countPerClass["weight"] = classes_weight
countPerClass

### Classes weigth for edge detection

In [ ]:
weight_edge_pixel = countPerEdge["edge_pixels"] / countPerEdge["total_pixels"]
weight_edge_pixel = weight_edge_pixel.to_numpy()
weight_edge_pixel

# Splitting dataset

In [ ]:
dataset_size = dataset.cardinality().numpy()
reduce_value = 1
#dataset = dataset.shuffle(dataset_size+1, seed=42)

reduced_dataset = dataset.take(dataset_size * reduce_value) if reduce_value != 1 else dataset
reduced_dataset_size = dataset_size * reduce_value

train_size = np.ceil(0.75 * reduced_dataset_size).astype(np.int32)
test_size = np.floor(0.25 * reduced_dataset_size).astype(np.int32)
val_size = test_size = int(0.5 * test_size)

train_dataset = reduced_dataset.take(train_size)
test_dataset = reduced_dataset.skip(train_size)
val_dataset = test_dataset.skip(val_size)
test_dataset = test_dataset.take(test_size)

train_size, test_size, val_size

# Data augmentation

In [ ]:
def data_transformation(image, mask):
    image_ = random_brightness(image, max_delta=0.12)
    image_ = random_contrast(image_, lower=0.9, upper=1)
    image_ = random_hue(image_, max_delta=0.09)
    image_ = random_jpeg_quality(image_, min_jpeg_quality=75, max_jpeg_quality=100)
    image_ = random_saturation(image_, lower=0.5, upper=1.5)

    aux = tf.concat([image_, mask], -1)

    aux_ = random_horizontal_flip(aux)
    aux_ = random_vertical_flip(aux_)

    return tf.cast(aux_[..., :3], tf.uint8), tf.cast(aux_[..., 3:], tf.uint8)

plt.subplots(3, 8,figsize=(15, 5))
for i in range(12):
    image_, mask_ = data_transformation(image, mask)

    plt.subplot(3, 8, (i*2)+1)
    plt.imshow(image_)
    plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)

    plt.subplot(3, 8, (i*2)+2)
    plt.imshow(mask_)
    plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)


# Applying the latest transformations

In [ ]:
def data_augmentation(image, mask_rgb, edge=False, train=True):
    image = tf.cast(image, dtype=tf.uint8)
    mask_rgb = tf.cast(mask_rgb, dtype=tf.uint8)

    if train == True:
        image, mask_rgb = data_transformation(image, mask_rgb)

    image = tf.reshape(image, shape=(size_img, size_img, 3))
    mask_rgb = tf.reshape(mask_rgb, shape=(size_img, size_img, 3))
    mask_ohe = rgb2ohe(mask_rgb, Labels.get_colors())

    image = tf.cast(image, dtype=tf.float32)/255
    mask_ohe = tf.cast(mask_ohe, dtype=tf.float32)

    if edge == True:
        edges = get_edges_from_mask(mask_rgb, rgb_format=False, lw=3)
        edges = tf.cast(edges, dtype=tf.float32)
        return image, {"segmentation": mask_ohe, "edge_detection_encoder": edges, "edge_detection_decoder": edges}
    else:
        return image, mask_ohe

def get_mask(image, mask_rgb):
    mask_rgb = tf.cast(mask_rgb, dtype=tf.uint8)
    mask_rgb = tf.reshape(mask_rgb, shape=(size_img, size_img, 3))
    mask_ohe = rgb2ohe(mask_rgb, Labels.get_colors())
    mask_ohe = tf.cast(mask_ohe, dtype=tf.float32)
    return mask_ohe

batch_size = 16

train_dataset_augment = train_dataset.map(lambda image, mask: data_augmentation(image=image, mask_rgb=mask, train=True), num_parallel_calls=autotune)
train_dataset_augment = train_dataset_augment.batch(batch_size)
train_dataset_augment = train_dataset_augment.prefetch(autotune)

val_dataset_augment = val_dataset.map(lambda image, mask: data_augmentation(image=image, mask_rgb=mask, train=False), num_parallel_calls=autotune)
val_dataset_augment = val_dataset_augment.batch(batch_size)
val_dataset_augment = val_dataset_augment.prefetch(autotune)

train_dataset_edge = train_dataset.map(lambda image, mask: data_augmentation(image=image, mask_rgb=mask, edge=True, train=True), num_parallel_calls=autotune)
train_dataset_edge = train_dataset_edge.batch(batch_size)
train_dataset_edge = train_dataset_edge.prefetch(autotune)

val_dataset_edge = val_dataset.map(lambda image, mask: data_augmentation(image=image, mask_rgb=mask, edge=True, train=False), num_parallel_calls=autotune)
val_dataset_edge = val_dataset_edge.batch(batch_size)
val_dataset_edge = val_dataset_edge.prefetch(autotune)

test_dataset_augment = test_dataset.map(lambda image, mask: data_augmentation(image=image, mask_rgb=mask, train=False), num_parallel_calls=autotune).batch(batch_size).prefetch(autotune)
test_dataset_edge = test_dataset.map(lambda image, mask: data_augmentation(image=image, mask_rgb=mask, edge=True, train=False), num_parallel_calls=autotune).batch(batch_size).prefetch(autotune)

## Plotting images and masks with all transformations

#### Only semantic segmentation task

In [ ]:
for trf_image, trf_mask_ohe in train_dataset_augment.shuffle(20).take(1).unbatch():
    plot_decomposition(image=tf.cast(trf_image*255, dtype=tf.int32), mask_ohe=trf_mask_ohe, df_colors=SSAI_colors, num_classes=num_classes)
    break

### For semantic segmentation task + edge detection task

In [ ]:
for trf_image, trf_output in train_dataset_edge.shuffle(20).take(1).unbatch():
    plot_decomposition(image=tf.cast(trf_image*255, dtype=tf.int32), mask_ohe=trf_output["segmentation"], edge=trf_output["edge_detection_encoder"], df_colors=SSAI_colors, num_classes=num_classes)
    break

# U-Net architecture from scratch

Reference: [U-Net: Convolutional Networks for Biomedical Image Segmentation](http://https://arxiv.org/abs/1505.04597)

<img src="https://i.ibb.co/93m575V/Sin-t-tulo.png" width=600>

## Encoder
<img src="https://i.ibb.co/yp1svTD/Encoder.png" width=300>


### How to calculate output size of convolution layers?
When padding parameters is setting to "**valid**":

The spatial size of the output volume is a function of the input volume size **W**, the kernel field size
**K** of the convolutional layer neurons, the stride **S**, and the amount of zero padding **P** on the border. The number of neurons that "fit" in a given volume is then:

$$\frac{W - K - 2P}{S} + 1$$

When padding parameters is setting to "**same**":

The output has the same size as the input

### How to calculate output size of pooling layers?
The resulting output, when using the "**valid**" padding option, has a spatial shape of:

$$Floor(\frac{W - K}{S}) + 1$$

Where K defined by pool_size.

But the resulting output shape when using the "**same**" padding option is:

$$ Floor(\frac{W - 1}{S}) + 1 $$

ResNet34, preprocess_input = Classifiers.get('resnet34')
backbone = ResNet34(input_shape=[256, 256, 3], weights='imagenet')
        
for i, layer in enumerate(backbone.layers):
        print(i, layer.name, layer.output.shape)

In [ ]:
@keras.saving.register_keras_serializable()
class Encoder(Model):
    def __init__(self, size_image=512, backbone=None, training_up_to_layer=35, mini_blocks=3, dropout=None, bn=False, initializer='he_normal', regularizer=None, name="Encoder", **kwargs):
        super().__init__(name=name, **kwargs)

        self.initializer = initializer
        self.regularizer = regularizer
        self.dropout = dropout
        self.state_backbone = backbone
        self.training_up_to_layer = training_up_to_layer
        self.bn = bn
        self.mini_blocks = mini_blocks
        self.size_image = size_image

        if self.state_backbone:
            self.preprocess_input, self.backbone = self.get_backbone()
        else:
            self.convs_block1, self.relus_block1, self.bns_block1, self.drop1_block1, self.mxp_block1 = self.get_block(filters=13, num_block=1)
            self.convs_block2, self.relus_block2, self.bns_block2, self.drop1_block2, self.mxp_block2 = self.get_block(filters=32, num_block=2)
            self.convs_block3, self.relus_block3, self.bns_block3, self.drop1_block3, self.mxp_block3 = self.get_block(filters=64, num_block=3)
            self.convs_block4, self.relus_block4, self.bns_block4, self.drop1_block4, self.mxp_block4 = self.get_block(filters=128, num_block=4)
            self.convs_block5, self.relus_block5, self.bns_block5, self.drop1_block5, self.mxp_block5 = self.get_block(filters=256, num_block=5)

        self.convs_block6, self.relus_block6, self.bns_block6, self.drop1_block6, self.mxp_block6 = self.get_block(filters=256, num_block=6)

    def get_block(self, filters=None, div=None, num_block=None):
        convs = []
        relus = []
        bns = []

        n=0
        for i in range(self.mini_blocks):
            convs.append(self.conv2d(filters=filters, name=f"en_block{num_block}_conv{n}"))
            if self.bn:
                bns.append(BatchNormalization(name=f"en_block{num_block}_bn{n}"))
            relus.append(ReLU(name=f"en_block{num_block}_relu{n}"))
            n += 1

            convs.append(self.conv2d(filters=filters, name=f"en_block{num_block}_conv{n}"))
            if self.bn:
                bns.append(BatchNormalization(name=f"en_block{num_block}_bn{n}"))
            relus.append(ReLU(name=f"en_block{num_block}_relu{n}"))
            n += 1

        mxp = MaxPooling2D(pool_size=(2, 2), strides=(2, 2), name=f"block{num_block}_mxp1")
        drop = Dropout(self.dropout, name=f"en_block{num_block}_drop1")
        return convs, relus, bns, drop, mxp

    def conv2d(self, filters, name):
        return Conv2D(filters=filters, kernel_size=(3, 3), activation=None, padding='same', strides=(1, 1), kernel_initializer=self.initializer, kernel_regularizer=self.regularizer, data_format="channels_last", name=name)

    def conv_block(self, _input, convs, relus, bns, drop, mxp):
        x = _input
        n=0

        for i in range(self.mini_blocks):
            skip = x = convs[n](x)
            x = relus[n](x)
            x = bns[n](x) if self.bn else x
            n += 1

            x = convs[n](x)
            x = relus[n](x)
            x = bns[n](x) if self.bn else x
            n += 1

            x = Add()([skip, x])

        x = Concatenate()([_input, x])

        block_output = x
        x = mxp(x)
        x = drop(x)
        return x, block_output

    def get_backbone(self):
        ResNet34, preprocess_input = Classifiers.get('resnet34')
        backbone = ResNet34(input_shape=[None, None, 3], weights='imagenet')

        for i, layer in enumerate(backbone.layers):
            if (i <= self.training_up_to_layer):
                layer.trainable = False
        #backbone.trainable = False

        #block1_output, block2_output, block3_output, block4_output = backbone.layers[27].output, backbone.layers[73].output, backbone.layers[141].output, backbone.layers[187].output
        block1_output, block2_output, block3_output, block4_output, block5_output, block6_output = backbone.layers[1].output, backbone.layers[5].output, backbone.layers[37].output, backbone.layers[74].output, backbone.layers[129].output, backbone.layers[157].output

        return preprocess_input, Model(inputs=[backbone.inputs], outputs=[block1_output, block2_output, block3_output, block4_output, block5_output, block6_output])

    def call(self, inputs, training=False):
        if self.state_backbone:
            x = self.preprocess_input(inputs)
            block1_output, block2_output, block3_output, block4_output, block5_output, block6_output = self.backbone(x, training=training)
            x = self.mxp_block6(block6_output)
            encoder_output = self.drop1_block6(x)
        else:
            # For input image shape: (256, 256, 3) and max_filters: 256
            x, block1_output = self.conv_block(_input=inputs, convs=self.convs_block1, relus=self.relus_block1, bns=self.bns_block1, mxp=self.mxp_block1, drop=self.drop1_block1) #x: (128, 128, 16), block1_output: (256, 256, 16)
            x, block2_output = self.conv_block(_input=x, convs=self.convs_block2, relus=self.relus_block2, bns=self.bns_block2, mxp=self.mxp_block2, drop=self.drop1_block2) #x: (64, 64, 64), block2_output: (128, 128, 64)
            x, block3_output = self.conv_block(_input=x, convs=self.convs_block3, relus=self.relus_block3, bns=self.bns_block3, mxp=self.mxp_block3, drop=self.drop1_block3) #x: (32, 32, 64), block3_output: (64, 64, 64)
            x, block4_output = self.conv_block(_input=x, convs=self.convs_block4, relus=self.relus_block4, bns=self.bns_block4, mxp=self.mxp_block4, drop=self.drop1_block4) #x: (16, 16, 128), block4_output: (32, 32, 128)
            x, block5_output = self.conv_block(_input=x, convs=self.convs_block5, relus=self.relus_block5, bns=self.bns_block5, mxp=self.mxp_block5, drop=self.drop1_block5) #x: (8, 8, 251), block5_output: (16, 16, 256)
            encoder_output, block6_output = self.conv_block(_input=x, convs=self.convs_block6, relus=self.relus_block6, bns=self.bns_block6, mxp=self.mxp_block6, drop=self.drop1_block6) #encoder_output: (4, 4, 507), block6_output: (8, 8, 512)

        return encoder_output, [block1_output, block2_output, block3_output, block4_output, block5_output, block6_output]

## Bottleneck

<img src="https://i.ibb.co/K9gpwPG/Bottleneck.png" width=500>

### How to calculate output size of upsampling layers?

$$ W_{ouput} = W_{input}.size[0]$$

$$ H_{ouput} = H_{input}.size[1]$$

### How to calculate output size of deconvolution layers?

When padding parameters is setting to "**valid**" and non-unit strides:

The spatial size of the output volume is a function of the input volume size **W**, the kernel field size
**K** of the convolutional layer neurons, the stride **S**, and the amount of zero padding **P** on the border. The number of neurons that "fit" in a given volume is then:

$$S.(W-1)+K$$

When padding parameters is setting to "**same**" and non-unit strides:

$$W.S$$

When padding parameters is setting to "**same**" and unit strides:

The output has the same size as the input.

In [ ]:
@keras.saving.register_keras_serializable()
class Bottleneck(Model):
    def __init__(self, max_filters=128, backbone=False, dropout=None, bn=False, initializer='he_normal', regularizer=None, name="Bottleneck", **kwargs):
        super().__init__(name=name, **kwargs)

        self.initializer = initializer
        self.regularizer = regularizer
        self.dropout = dropout
        self.bn = bn

        self.bn1 = BatchNormalization(name="btn_bn1")
        self.bn2 = BatchNormalization(name="btn_bn2")
        self.btn_conv1 = self.conv2d(filters=max_filters, name="btn_conv1")
        self.btn_conv2 = self.conv2d(filters=max_filters, name="btn_conv2")
        self.relu1 = ReLU(name="btn_relu1")
        self.relu2 = ReLU(name="btn_relu2")
        self.bn3 = BatchNormalization(name="btn_bn3")
        self.bn4 = BatchNormalization(name="btn_bn4")
        self.btn_conv3 = self.conv2d(filters=max_filters*2, name="btn_conv3")
        self.btn_conv4 = self.conv2d(filters=512 if backbone else 752, name="btn_conv4")
        self.relu3 = ReLU(name="btn_relu3")
        self.relu4 = ReLU(name="btn_relu4")

        self.btn_drop1 = Dropout(self.dropout, name="btn_drop1")
        self.btn_upconv1 = self.conv2dtranspose(filters=512 if backbone else 752, name="btn_upconv1")

    def conv2d(self, filters, name):
        return Conv2D(filters=filters, kernel_size=(3, 3), activation=None, padding='same', strides=(1, 1), kernel_initializer=self.initializer, kernel_regularizer=self.regularizer, data_format="channels_last", name=name)

    def conv2dtranspose(self, filters, name):
        return Conv2DTranspose(filters=filters, kernel_size=(2, 2), strides=(2, 2), activation="relu", kernel_initializer=self.initializer, kernel_regularizer=self.regularizer, data_format="channels_last", name=name)

    def call(self, inputs, training=False):

        # For input image shape: (256, 256, 3) and max_filters: 256
        skip = x = self.btn_conv1(inputs) # x: (4, 4, 256)
        x = self.relu1(x)
        x = self.bn1(x) if self.bn else x
        x = self.btn_conv2(x) # x: (4, 4, 256)
        x = self.relu2(x)
        x = self.bn2(x) if self.bn else x
        x = Add()([skip, x])

        x = self.btn_conv3(x) # x: (4, 4, 256)
        x = self.relu3(x)
        x = self.bn3(x) if self.bn else x
        x = self.btn_conv4(x) # x: (4, 4, 256)
        x = self.relu4(x)
        x = self.bn4(x) if self.bn else x
        x = Add()([inputs, x])

        x = self.btn_upconv1(x) # bottleneck_output: (8, 8, 128)
        bottleneck_output = self.btn_drop1(x)

        return bottleneck_output

## Decoder

<img src="https://i.ibb.co/xjzZBv2/Sin-t-tulo3.png" width=600>

In [ ]:
@keras.saving.register_keras_serializable()
class Decoder(Model):
    def __init__(self, mini_blocks=3, backbone=False, dropout=None, bn=False, initializer='he_normal', regularizer=None, name="Decoder", **kwargs):
        super().__init__(name=name, **kwargs)

        self.initializer = initializer
        self.regularizer = regularizer
        self.dropout = dropout
        self.bn = bn
        self.mini_blocks = mini_blocks

        self.convs_block1, self.relus_block1, self.bns_block1, self.drop1_block1, self.upconv1_block1 = self.get_block(filters=512 if backbone else 752, filters_up=256 if backbone else 496, num_block=1)
        self.convs_block2, self.relus_block2, self.bns_block2, self.drop1_block2, self.upconv1_block2 = self.get_block(filters=256 if backbone else 496, filters_up=128 if backbone else 240, num_block=2)
        self.convs_block3, self.relus_block3, self.bns_block3, self.drop1_block3, self.upconv1_block3 = self.get_block(filters=128 if backbone else 240, filters_up=64 if backbone else 112, num_block=3)
        self.convs_block4, self.relus_block4, self.bns_block4, self.drop1_block4, self.upconv1_block4 = self.get_block(filters=64 if backbone else 112, filters_up=64 if backbone else 48, num_block=4)
        self.convs_block5, self.relus_block5, self.bns_block5, self.drop1_block5, self.upconv1_block5 = self.get_block(filters=64 if backbone else 48, filters_up=3 if backbone else 16, num_block=5)
        self.convs_block6, self.relus_block6, self.bns_block6, self.drop1_block6, _ = self.get_block(filters=3 if backbone else 16, filters_up=16 if backbone else 16, num_block=6)
        self.bn_block7 = BatchNormalization(name=f"de_block7_bn1")
        self.conv1_block7 = self.conv2d(16, name="de_block7_conv1")
        self.relu1_block7 = ReLU(name="de_block7_relu1")
        self.conv2_block7 = Conv2D(filters=num_classes, data_format="channels_last", kernel_size=(1, 1), activation="softmax", padding='same', kernel_initializer=self.initializer, kernel_regularizer=self.regularizer, strides=(1, 1), name="decoder_output")

    def get_block(self, filters, filters_up, num_block):
        convs = []
        bn = []
        relus = []
        n=0

        for i in range(self.mini_blocks):
            convs.append(self.conv2d(filters=filters, name=f"de_block{num_block}_conv{n}"))
            if self.bn:
                bn.append(BatchNormalization(name=f"de_block{num_block}_bn{n}"))
            relus.append(ReLU(name=f"de_block{num_block}_relu{n}"))
            n += 1

            convs.append(self.conv2d(filters=filters, name=f"de_block{num_block}_conv{n}"))
            if self.bn:
                bn.append(BatchNormalization(name=f"de_block{num_block}_bn{n}"))
            relus.append(ReLU(name=f"de_block{num_block}_relu{n}"))
            n += 1

        upconv1 = self.conv2dtranspose(filters=filters_up, name=f"de_block{num_block}_upconv1")
        drop_ = Dropout(self.dropout, name=f"de_block{num_block}_drop1")
        return convs, relus, bn, drop_, upconv1

    def conv2d(self, filters=64, kernel_size=(3, 3), strides=(1, 1), name=None):
        return Conv2D(filters=filters, kernel_size=kernel_size, activation=None, padding='same', strides=strides, kernel_initializer=self.initializer, kernel_regularizer=self.regularizer, data_format="channels_last", name=name)

    def conv2dtranspose(self, filters=64, kernel_size=(2, 2), strides=(2, 2), name=None):
        return Conv2DTranspose(filters=filters, kernel_size=kernel_size, strides=strides, kernel_initializer=self.initializer, kernel_regularizer=self.regularizer, data_format="channels_last", name=name)

    def conv_block(self, _input, skip_connection, convs, relus, bns, drop, upconv1=None):
        x = Add()([skip_connection, _input])
        input_skip = x
        n=0

        for i in range(self.mini_blocks):
            skip = x = convs[n](x)
            x = relus[n](x)
            x = bns[n](x) if self.bn else x
            n += 1

            x = convs[n](x)
            x = relus[n](x)
            x = bns[n](x) if self.bn else x
            n += 1

            x = Add()([skip, x])

        x = Add()([input_skip, x])
        de_block_output = x

        if upconv1:
            x = upconv1(x)

        x = drop(x)

        return x, de_block_output

    def call(self, inputs, skip_connections, training=False):

        # For input image shape: (256, 256, 3) and max_filters: 256
        x, de_block1_output = self.conv_block(_input=inputs, skip_connection=skip_connections[-1], convs=self.convs_block1, relus=self.relus_block1, bns=self.bns_block1, drop=self.drop1_block1, upconv1=self.upconv1_block1) # x: (16, 16, 891) de_block1_output: (8, 8, 891)
        x, de_block2_output = self.conv_block(_input=x, skip_connection=skip_connections[-2], convs=self.convs_block2, relus=self.relus_block2, bns=self.bns_block2, drop=self.drop1_block2, upconv1=self.upconv1_block2) # x: (32, 32, 1270) de_block2_output: (16, 16, 1270)
        x, de_block3_output = self.conv_block(_input=x, skip_connection=skip_connections[-3], convs=self.convs_block3, relus=self.relus_block3, bns=self.bns_block3, drop=self.drop1_block3, upconv1=self.upconv1_block3) # x: (64, 64, 1457) de_block3_output: (32, 32, 1457)
        x, de_block4_output = self.conv_block(_input=x, skip_connection=skip_connections[-4], convs=self.convs_block4, relus=self.relus_block4, bns=self.bns_block4, drop=self.drop1_block4, upconv1=self.upconv1_block4) # x: (128, 128, 1548) de_block4_output: (64, 64, 1548)
        x, de_block5_output = self.conv_block(_input=x, skip_connection=skip_connections[-5], convs=self.convs_block5, relus=self.relus_block5, bns=self.bns_block5, drop=self.drop1_block5, upconv1=self.upconv1_block5) # x: (256, 256, 1607) de_block5_output: (128, 128, 1607)
        _, de_block6_output = self.conv_block(_input=x, skip_connection=skip_connections[-6], convs=self.convs_block6, relus=self.relus_block6, bns=self.bns_block6, drop=self.drop1_block6) # de_block6_output: (256, 256, 1634)

        x = self.conv1_block7(de_block6_output)
        x = self.relu1_block7(x)
        x = self.bn_block7(x) if self.bn else x
        decoder_output = self.conv2_block7(x)

        return decoder_output, [de_block1_output, de_block2_output, de_block3_output, de_block4_output, de_block5_output, de_block6_output]

## Edge Loss Reinforced Semantic Segmentation Network (ERN)

Reference: [ERN: Edge Loss Reinforced Semantic Segmentation Network for Remote Sensing Images](https://www.mdpi.com/2072-4292/10/9/1339)

In [ ]:
@keras.saving.register_keras_serializable()
class ERN(Model):
    def __init__(self, backbone=False, bn=False, initializer='he_normal', regularizer=None, name="ERN", **kwargs):
        super().__init__(name=name, **kwargs)

        self.initializer = initializer
        self.regularizer = regularizer
        self.backbone = backbone
        self.bn = bn

        self.enc_edge_block6_upconv1 = self.conv2dtranspose(filters=256 if backbone else 496, name=f"enc_edge_block5_upconv1")
        self.enc_edge_block5_upconv1 = self.conv2dtranspose(filters=128 if backbone else 240, name=f"enc_edge_block4_upconv1")
        self.enc_edge_block4_upconv1 = self.conv2dtranspose(filters=64 if backbone else 112, name=f"enc_edge_block3_upconv1")
        self.enc_edge_block3_upconv1 = self.conv2dtranspose(filters=64 if backbone else 48, name=f"enc_edge_block2_upconv1")
        self.enc_edge_block2_upconv1 = self.conv2dtranspose(filters=3 if backbone else 16, name=f"enc_edge_block1_upconv1")
        self.enc_edge_conv1 = self.conv2d(filters=128, name=f"enc_edge_conv1")
        self.enc_edge_bn1 = BatchNormalization(name=f"enc_edge_bn1")
        self.enc_edge_relu1 = ReLU(name=f"enc_edge_relu1")
        self.enc_edge_conv2 = self.conv2d(filters=128, name=f"enc_edge_conv2")
        self.enc_edge_bn2 = BatchNormalization(name=f"enc_edge_bn2")
        self.enc_edge_relu2 = ReLU(name=f"enc_edge_relu2")
        self.enc_edge_conv3 = self.conv2d(filters=64, name=f"enc_edge_conv3")
        self.enc_edge_bn3 = BatchNormalization(name=f"enc_edge_bn3")
        self.enc_edge_relu3 = ReLU(name=f"enc_edge_relu3")
        self.enc_edge_conv4 = Conv2D(filters=1, data_format="channels_last", kernel_size=(2, 2), activation="sigmoid", padding='same', kernel_initializer=self.initializer, kernel_regularizer=self.regularizer, strides=(1, 1), name=f"enc_edge_conv4")

        self.dec_edge_block5_upconv1 = self.conv2dtranspose(filters=3 if backbone else 16, name=f"dec_edge_block5_upconv1")
        self.dec_edge_block4_upconv1 = self.conv2dtranspose(filters=64 if backbone else 48, name=f"dec_edge_block4_upconv1")
        self.dec_edge_block3_upconv1 = self.conv2dtranspose(filters=64 if backbone else 112, name=f"dec_edge_block3_upconv1")
        self.dec_edge_block2_upconv1 = self.conv2dtranspose(filters=128 if backbone else 240, name=f"dec_edge_block2_upconv1")
        self.dec_edge_block1_upconv1 = self.conv2dtranspose(filters=256 if backbone else 496, name=f"dec_edge_block1_upconv1")
        self.dec_edge_conv1 = self.conv2d(filters=128, name=f"dec_edge_conv1")
        self.dec_edge_bn1 = BatchNormalization(name=f"dec_edge_bn1")
        self.dec_edge_relu1 = ReLU(name=f"dec_edge_relu1")
        self.dec_edge_conv2 = self.conv2d(filters=128, name=f"dec_edge_conv2")
        self.dec_edge_bn2 = BatchNormalization(name=f"dec_edge_bn2")
        self.dec_edge_relu2 = ReLU(name=f"dec_edge_relu2")
        self.dec_edge_conv3 = self.conv2d(filters=64, name=f"dec_edge_conv3")
        self.dec_edge_bn3 = BatchNormalization(name=f"dec_edge_bn3")
        self.dec_edge_relu3 = ReLU(name=f"dec_edge_relu3")
        self.dec_edge_conv4 = Conv2D(filters=1, data_format="channels_last", kernel_size=(2, 2), activation="sigmoid", padding='same', kernel_initializer=self.initializer, kernel_regularizer=self.regularizer, strides=(1, 1), name=f"dec_edge_conv4")

    def conv2d(self, filters=64, kernel_size=(3, 3), strides=(1, 1), name=None):
        return Conv2D(filters=filters, kernel_size=kernel_size, activation=None, padding='same', strides=strides, kernel_initializer=self.initializer, kernel_regularizer=self.regularizer, data_format="channels_last", name=name)

    def conv2dtranspose(self, filters=64, activation="relu", kernel_size=(2, 2), strides=(2, 2), name=None):
        return Conv2DTranspose(filters=filters, kernel_size=kernel_size, activation=activation, strides=strides, kernel_initializer=self.initializer, kernel_regularizer=self.regularizer, data_format="channels_last", name=name)

    def call(self, encoder_outputs, decoder_outputs, training=False):
        # input: (8, 8, -)
        x = self.enc_edge_block6_upconv1(encoder_outputs[-1]) # x: (16, 16, -)
        x = Add()([encoder_outputs[-2], x])
        x = self.enc_edge_block5_upconv1(x) # x: (32, 32, -)
        x = Add()([encoder_outputs[-3], x])
        x = self.enc_edge_block4_upconv1(x) # x: (64, 64, -)
        x = Add()([encoder_outputs[-4], x])
        x = self.enc_edge_block3_upconv1(x) # x: (128, 128, -)
        x = Add()([encoder_outputs[-5], x])
        x = self.enc_edge_block2_upconv1(x) # x: (256, 256, -)
        x = Add()([encoder_outputs[-6], x])
        x = self.enc_edge_conv1(x) # x: (256, 256, -)
        x = self.enc_edge_bn1(x) if self.bn else x
        x = self.enc_edge_relu1(x)
        x = self.enc_edge_conv2(x) # x: (256, 256, -)
        x = self.enc_edge_bn2(x) if self.bn else x
        x = self.enc_edge_relu2(x)
        x = self.enc_edge_conv3(x) # x: (256, 256, -)
        x = self.enc_edge_bn3(x) if self.bn else x
        x = self.enc_edge_relu3(x)
        enc_edge_output = self.enc_edge_conv4(x) # x: (256, 256, 1)

        # input: (8, 8, -)
        y = self.dec_edge_block1_upconv1(decoder_outputs[0]) # x: (16, 16, -)
        y = Add()([decoder_outputs[1], y])
        y = self.dec_edge_block2_upconv1(y) # x: (32, 32, -)
        y = Add()([decoder_outputs[2], y])
        y = self.dec_edge_block3_upconv1(y) # x: (64, 64, -)
        y = Add()([decoder_outputs[3], y])
        y = self.dec_edge_block4_upconv1(y) # x: (128, 128, -)
        y = Add()([decoder_outputs[4], y])
        y = self.dec_edge_block5_upconv1(y) # x: (256, 256, -)
        y = Add()([decoder_outputs[5], y])
        y = self.dec_edge_conv1(y) # x: (256, 256, -)
        y = self.dec_edge_bn1(y) if self.bn else y
        y = self.dec_edge_relu1(y)
        y = self.dec_edge_conv2(y) # x: (256, 256, -)
        y = self.dec_edge_bn2(y) if self.bn else y
        y = self.dec_edge_relu2(y)
        y = self.dec_edge_conv3(y) # x: (256, 256, -)
        y = self.dec_edge_bn3(y) if self.bn else y
        y = self.dec_edge_relu3(y)
        dec_edge_output = self.dec_edge_conv4(y) # x: (256, 256, 1)

        return enc_edge_output, dec_edge_output

## U-Net model
Besides, this section we enable to the network to replace encoder from scratch for adapt a backbone e.g. Resnet50, Resnet34 or VGG, this we enable to perform techniques like Transfer learning or Fine tuning.

In [ ]:
@keras.saving.register_keras_serializable()
class MyUnetModel(Model):
    def __init__(self, num_classes=7, size_image=512, backbone=False, training_up_to_layer=37, mini_blocks=3, ern=False, dropout=None, batch_norm=False, initializer='he_normal', regularizer=None, name="MyUnetModel", **kwargs):
        super().__init__(name=name, **kwargs)
        self.size_image = size_image
        self.ern = ern

        self.encoder = Encoder(size_image=size_image, backbone=backbone, training_up_to_layer=training_up_to_layer, mini_blocks=mini_blocks, dropout=dropout, bn=batch_norm, initializer=initializer, regularizer=regularizer)
        self.bottleneck = Bottleneck(backbone=backbone, dropout=dropout, bn=batch_norm, initializer=initializer, regularizer=regularizer)
        self.decoder = Decoder(backbone=backbone, mini_blocks=mini_blocks, dropout=dropout, bn=batch_norm, initializer=initializer, regularizer=regularizer)
        self.ern_net = ERN(backbone=backbone, bn=batch_norm, initializer=initializer, regularizer=regularizer)

    def call(self, inputs):
        encoder_output, skip_connections_encoder = self.encoder(inputs)
        bottleneck_output = self.bottleneck(encoder_output)
        decoder_output, skip_connections_decoder = self.decoder(bottleneck_output, skip_connections_encoder)
        if self.ern:
            enc_edge_output, dec_edge_output = self.ern_net(skip_connections_encoder, skip_connections_decoder)
            return {"segmentation": decoder_output, "edge_detection_encoder": enc_edge_output, "edge_detection_decoder": dec_edge_output}
        else:
            return decoder_output

    def build_graph(self):
        x = Input(shape=(self.size_image, self.size_image, 3))
        return Model(inputs=[x], outputs=self.call(x))

## Creating a instance of the model

In [ ]:
max_filters = 256
unet_model = MyUnetModel(num_classes=num_classes,
                         size_image=size_img,
                         backbone=True,
                         ern=True,
                         dropout=0.2,
                         batch_norm=True,
                         regularizer=L2(l2=1e-5)
                        )
unet_model.build(input_shape=(32, 256, 256, 3))
unet_model.summary(expand_nested=True)

## Plotting the model architecture

I drew the model architecture in Latex with the tool [PlotNeuralNet](https://github.com/HarisIqbal88/PlotNeuralNet)

In [ ]:
doc = fitz.open("/kaggle/working/PlotNeuralNet/unet.pdf")

for page_number in range(doc.page_count):
    page = doc[page_number]
    image = page.get_pixmap()
    image_path = "/kaggle/working/unet_ern_model.jpg"
    image.save(image_path)

doc.close()

PIL.Image.open("/kaggle/working/unet_ern_model.jpg")

# Dice loss and Dice coefficient for image segmentation

In [ ]:
def dice_coefficient(y_true, y_pred):
    intersection = K.sum(K.abs(y_true * y_pred), axis=[3, 2, 1])
    dn = K.sum(K.square(y_true) + K.square(y_pred), axis=[3, 2, 1]) + epsilon()
    return K.mean(2 * intersection / dn)

def dice_loss(y_true, y_pred):
    intersection = K.sum(K.abs(y_true * y_pred), axis=[3, 2, 1])
    dn = K.sum(K.square(y_true) + K.square(y_pred), axis=[3, 2, 1]) + epsilon()
    dl = 2 * intersection / dn
    return - K.mean(dl)

# Looking for optimal learning rate

## Defining useful functions for training the model

### Callbalck to display the change in the parameters during training

In [ ]:
def show_parameters(parameters=None):
    df_lr = pd.read_csv("/kaggle/working/learning_rate_scheduler.csv")
    fig, axs = plt.subplots(2, 4, figsize=(25, 8))

    for i, col in enumerate(parameters):
        plt.subplot(2, 4, i+1)
        param = col[0].replace('_', ' ').title()
        plt.semilogx(df_lr["lr"], df_lr[col[0]], label=f"Training {param}")
        plt.semilogx(df_lr["lr"], df_lr[col[1]], label=f"Validation {param}")
        plt.ylabel(col[0].capitalize().replace("_", " "))
        plt.xlabel("Learning Rate")
        plt.legend(fontsize=8)
        plt.grid(True, alpha=0.3, which="both")
        plt.title(f"Learning Rate vs {param}")

    fig, axs = delete_empty_subplots(fig, axs)
    plt.tight_layout()
    plt.show()
    return

class DisplayParameters(Callback):
    def on_epoch_end(self, epoch, logs=None):
        clear_output(wait=True)
        show_parameters(parameters=[["loss", "val_loss"],
                                    #["segmentation_loss", "val_segmentation_loss"],
                                    #["edge_detection_encoder_loss", "val_edge_detection_encoder_loss"],
                                    #["edge_detection_decoder_loss", "val_edge_detection_decoder_loss"],
                                    #["segmentation_dice_coefficient", "val_segmentation_dice_coefficient"],
                                    ["dice_coefficient", "val_dice_coefficient"],
                                    ["one_hot_io_u", "val_one_hot_io_u"],
                                    ["categorical_accuracy", "val_categorical_accuracy"],
                                    #["edge_detection_encoder_binary_crossentropy", "val_edge_detection_encoder_binary_crossentropy"],
                                    #["edge_detection_decoder_binary_crossentropy", "val_edge_detection_decoder_binary_crossentropy"]
                                   ])
        print ('\nParameters after epoch {}\n'.format(epoch+1))
        return

### Callaback to save the epoch result into CSV file

In [ ]:
csvlogger = tf.keras.callbacks.CSVLogger(dst_path + f"learning_rate_scheduler.csv", separator=',', append=True)

## Cyclical Learning Rate

Reference: [Cyclical Learning Rates for Training Neural Networks](https://arxiv.org/abs/1506.01186)

This method is based on the fact that a varying learning rate produce a improve in the process of model training, e.g. better scores are achieved in a shorter training time than the standard method.

>This paper demonstrates the surprising phenomenon that a varying learning rate during training is beneficial overall and thus proposes to let the global learning rate vary cyclically within a band of values instead of setting it to a fixed value.

>The potential benefits of CLR can be seen in Figure 1, which shows the test data classification accuracy of the
CIFAR-10 dataset during training1. The baseline (blue curve) reaches a final accuracy of 81.4% after 70, 000 iterations. In contrast, it is possible to fully train the network using the CLR method instead of tuning (red curve) within 25,000 iterations and attain the same accuracy.<br><img src="https://i.postimg.cc/PrkVtnn4/image.png" width=300><br><cite style="font-size:10px">Figure 1. Classification accuracy while training CIFAR-10. The red curve shows the result of training with one of the new learning rate policies.</cite>

>The essence of this learning rate policy comes from the observation that increasing the learning rate might have a short term negative effect and yet achieve a longer term beneficial effect. This observation leads to the idea of letting the learning rate vary within a range of values rather than adopting a stepwise fixed or exponentially decreasing value. That is, **one sets minimum and maximum boundaries and the learning rate cyclically varies between these bounds**. Experiments with numerous functional forms, such as a triangular window (linear), a Welch window (parabolic) and a Hann window (sinusoidal) all produced equivalent results.

In my case, I adopted a triangular learning rate policy, similar to the below figure:

<img src="https://i.postimg.cc/T3dkDjw5/image.png"  width=280>

#### How can one estimate a good value for the cycle length?
> The length of a cycle and the input parameter stepsize can be easily computed from the number of iterations in an ***epoch***. An ***epoch*** is calculated by dividing the number of training images by the ***batchsize*** used.

> The final accuracy results are actually quite robust to cycle length but experiments show that it often is good to set ***stepsize*** equal to 2 − 10 times the number of iterations in an epoch.

####  How can one estimate reasonable minimum and maximum boundary values?
> There is a simple way to estimate reasonable ***minimum and maximum boundary values*** with one training run of the network for a few epochs. It is a “***LR range test***”; run your model for several epochs while letting the learning rate increase linearly between low and high ***LR*** values.

In [ ]:
class CyclicLR(Callback):
    def __init__(self, base_lr=0.001, max_lr=0.006, step_size=2000., mode='triangular',
                 gamma=1., scale_fn=None, scale_mode='cycle'):
        super(CyclicLR, self).__init__()

        self.base_lr = base_lr
        self.max_lr = max_lr
        self.step_size = step_size
        self.mode = mode
        self.gamma = gamma
        if scale_fn == None:
            if self.mode == 'triangular':
                self.scale_fn = lambda x: 1.
                self.scale_mode = 'cycle'
            elif self.mode == 'triangular2':
                self.scale_fn = lambda x: 1/(2.**(x-1))
                self.scale_mode = 'cycle'
            elif self.mode == 'exp_range':
                self.scale_fn = lambda x: gamma**(x)
                self.scale_mode = 'iterations'
        else:
            self.scale_fn = scale_fn
            self.scale_mode = scale_mode
        self.clr_iterations = 0.
        self.trn_iterations = 0.
        self.history = {}

        self._reset()

    def _reset(self, new_base_lr=None, new_max_lr=None,
               new_step_size=None):
        """Resets cycle iterations.
        Optional boundary/step size adjustment.
        """
        if new_base_lr != None:
            self.base_lr = new_base_lr
        if new_max_lr != None:
            self.max_lr = new_max_lr
        if new_step_size != None:
            self.step_size = new_step_size
        self.clr_iterations = 0.

    def clr(self):
        cycle = np.floor(1+self.clr_iterations/(2*self.step_size))
        x = np.abs(self.clr_iterations/self.step_size - 2*cycle + 1)
        if self.scale_mode == 'cycle':
            return self.base_lr + (self.max_lr-self.base_lr)*np.maximum(0, (1-x))*self.scale_fn(cycle)
        else:
            return self.base_lr + (self.max_lr-self.base_lr)*np.maximum(0, (1-x))*self.scale_fn(self.clr_iterations)

    def on_train_begin(self, logs={}):
        logs = logs or {}

        if self.clr_iterations == 0:
            K.set_value(self.model.optimizer.lr, self.base_lr)
        else:
            K.set_value(self.model.optimizer.lr, self.clr())

    def on_batch_end(self, epoch, logs=None):

        logs = logs or {}
        self.trn_iterations += 1
        self.clr_iterations += 1

        self.history.setdefault('lr', []).append(K.get_value(self.model.optimizer.lr))
        self.history.setdefault('iterations', []).append(self.trn_iterations)

        for k, v in logs.items():
            self.history.setdefault(k, []).append(v)

        K.set_value(self.model.optimizer.lr, self.clr())

### Defining a scheduler to perform the LR range test

In [ ]:
def scheduler(epoch, lr_init):
     return lr_init * 10**(epoch/4.5)

######################
lr_init = 1e-9
epochs = 30
#####################

lr_sch = LearningRateScheduler(lambda epoch: scheduler(epoch, lr_init))

#To show the learning tracking line
x = range(epochs)
y = [scheduler(epoch, lr_init) for epoch in x]

plt.subplots(1, 2, figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(x, y, marker='o', markersize=3)
plt.xlabel("Epochs")
plt.ylabel("Learning rate")
plt.title("Learning rate tracking")
plt.grid(which="both", alpha=0.3, linestyle=':')
plt.minorticks_on()

plt.subplot(1, 2, 2)
plt.plot(x, y, marker='o', markersize=3)
plt.yscale("log")
plt.xlabel("Epochs")
plt.ylabel("Learning rate")
plt.title("Learning rate tracking (Log Scale)")
plt.grid(which="both", alpha=0.3, linestyle=':')
plt.minorticks_on()

import shutil
import os
import tensorflow as tf

os.remove(f"/kaggle/working/learning_rate_scheduler.csv")

## Training the model for the LR range test

In [ ]:
tf.keras.backend.clear_session()

#Create a instance of my Unet class
max_filters = 256
unet_model = MyUnetModel(num_classes=num_classes,
                         size_image=size_img,
                         backbone=False,
                         ern=False,
                         dropout=0.2,
                         batch_norm=True,
                         regularizer=L2(l2=1e-5)
                        )

#Compiling the model
unet_model.compile(loss=dice_loss,
                   optimizer=Adam(learning_rate=lr_init),
                   metrics=[dice_coefficient,
                            OneHotIoU(num_classes=num_classes, target_class_ids=[i for i in range(num_classes)]),
                            CategoricalAccuracy()])

#Training the model or showing the results in the parameters
if not os.path.exists(dst_path+"learning_rate_scheduler.csv"):
    history = unet_model.fit(train_dataset_augment,
                             validation_data=val_dataset_augment,
                             epochs=epochs,
                             callbacks=[
                                 lr_sch,
                                 csvlogger,
                                 DisplayParameters()
                             ],
                             batch_size=batch_size,
                             validation_batch_size=batch_size,
                             verbose=1,
                             class_weight=dict(zip([0, 1, 2, 3], classes_weight))
                            )
else:
    show_parameters(parameters=[["loss", "val_loss"],
                                #["segmentation_loss", "val_segmentation_loss"],
                                #["edge_detection_encoder_loss", "val_edge_detection_encoder_loss"],
                                #["edge_detection_decoder_loss", "val_edge_detection_decoder_loss"],
                                #["segmentation_dice_coefficient", "val_segmentation_dice_coefficient"],
                                ["dice_coefficient", "val_dice_coefficient"],
                                ["one_hot_io_u", "val_one_hot_io_u"],
                                ["categorical_accuracy", "val_categorical_accuracy"],
                                #["edge_detection_encoder_binary_crossentropy", "val_edge_detection_encoder_binary_crossentropy"],
                                #["edge_detection_decoder_binary_crossentropy", "val_edge_detection_decoder_binary_crossentropy"]
                               ])

We can conclude that a good range to **LR parameter** could be between **1e-6 to 8e-5**.

In [ ]:
lr_init = 1e-6
max_lr = 8e-5

# Training the model

## Defining useful tools for training the model

### Callback to display the advance in the predictions during training

In [ ]:
def show_predictions(dataset_augment=None, df_colors=SSAI_colors, edge=True):
    for trf_image, _ in dataset_augment.shuffle(200).take(1):
        pred = unet_model.predict(trf_image)

        if edge:
            pred_mask = tf.math.argmax(pred["segmentation"][0], axis=-1)
            pred_mask = tf.one_hot(pred_mask, depth=num_classes)
            pred_edge_enc = tf.math.argmax(pred["edge_detection_encoder"][0], axis=-1)
            pred_edge_dec = (pred['edge_detection_decoder'][0] > 0.5)+0.
            plot_decomposition(image=tf.cast(trf_image[0]*255, dtype=tf.int32), mask_ohe=pred_mask, edge=pred_edge_dec, df_colors=df_colors, num_classes=num_classes)
        else:
            pred_mask = tf.math.argmax(pred[0], axis=-1)
            pred_mask = tf.one_hot(pred_mask, depth=num_classes)
            plot_decomposition(image=tf.cast(trf_image[0]*255, dtype=tf.int32), mask_ohe=pred_mask, df_colors=df_colors, num_classes=num_classes)

        plt.show()
        break
    return

class DisplayCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        clear_output(wait=True)
        show_predictions(dataset_augment=val_dataset_edge)
        print ('\nSample Prediction after epoch {}\n'.format(epoch+1))
        return

class DisplayCallback_baseline(Callback):
    def on_epoch_end(self, epoch, logs=None):
        clear_output(wait=True)
        show_predictions(dataset_augment=val_dataset_augment, edge=False)
        print ('\nSample Prediction after epoch {}\n'.format(epoch+1))
        return

### Setting the scheduler for learning rate

In [ ]:
#reduce_lr = ReduceLROnPlateau(monitor = 'val_loss', factor=0.5, patience=2)
clr = CyclicLR(base_lr=lr_init, max_lr=max_lr, step_size=1354.0*2, mode='triangular')

### Callback to stop training when a monitored metric has stopped improving

In [ ]:
early_stop = EarlyStopping(monitor = 'val_loss', patience=5)

### Callback to save the epoch result into CSV file

In [ ]:
version_unet_ern = "_ern" #Model version that we are training
csvlogger_unet_ern = CSVLogger(dst_path + f"history{version_unet_ern}.csv", separator=',', append=True)

version_baseline = "_baseline"
csvlogger_baseline = CSVLogger(dst_path + f"history{version_baseline}.csv", separator=',', append=True)

version_unet = "_only_unet"
csvlogger_unet = CSVLogger(dst_path + f"history{version_unet}.csv", separator=',', append=True)

### Callback to save the Keras model at some frequency

In [ ]:
checkpoint_filepath_unet_ern = dst_path + f'modelckpt{version_unet_ern}/checkpoint_basic_unet.ckpt'
ckpt_callback_unet_ern = ModelCheckpoint(
    filepath=checkpoint_filepath_unet_ern,
    monitor='val_loss',
    mode='min',
    save_best_only=True
)

checkpoint_filepath_baseline = dst_path + f'modelckpt{version_baseline}/checkpoint_basic_unet.ckpt'
ckpt_callback_baseline = ModelCheckpoint(
    filepath=checkpoint_filepath_baseline,
    monitor='val_loss',
    mode='min',
    save_best_only=True
)

checkpoint_filepath_unet = dst_path + f'modelckpt{version_unet}/checkpoint_basic_unet.ckpt'
ckpt_callback_unet = ModelCheckpoint(
    filepath=checkpoint_filepath_unet,
    monitor='val_loss',
    mode='min',
    save_best_only=True
)

## Let's train our models

### Baseline U-NET

import shutil

shutil.rmtree(checkpoint_filepath_baseline)
os.remove(f"history{version_baseline}.csv")

In [ ]:
epochs = 100
max_filters = 256

unet_model = MyUnetModel(num_classes=num_classes,
                         size_image=size_img,
                         backbone=False,
                         ern=False,
                         dropout=0.2,
                         batch_norm=True,
                         regularizer=L2(l2=1e-5)
                        )

#Compiling the model
unet_model.compile(loss=dice_loss,
                   optimizer=Adam(learning_rate=lr_init),
                   metrics=[dice_coefficient,
                            OneHotIoU(num_classes=num_classes, target_class_ids=[i for i in range(num_classes)]),
                            CategoricalAccuracy()])

#Loading the most recent saved model weights to continue the training
if os.path.exists(checkpoint_filepath_baseline):
    print("Loading...")
    unet_model.load_weights(checkpoint_filepath_baseline)

#Training the model
history = unet_model.fit(train_dataset_augment,
                         validation_data=val_dataset_augment,
                         epochs=epochs,
                         callbacks=[
                             early_stop,
                             ckpt_callback_baseline,
                             clr,
                             csvlogger_baseline,
                             DisplayCallback_baseline()
                         ],
                         batch_size=batch_size,
                         validation_batch_size=batch_size,
                         verbose=1,
                         class_weight=dict(zip([0, 1, 2, 3], classes_weight))
                        )

### U-NET with ResNet34 as encoder

In [ ]:
epochs = 100
#backbone.layers[1].output, backbone.layers[5].output, backbone.layers[37].output, backbone.layers[74].output, backbone.layers[129].output, backbone.layers[157].output

unet_model = MyUnetModel(num_classes=num_classes,
                         size_image=size_img,
                         backbone=True,
                         training_up_to_layer=37,
                         mini_blocks=3,
                         ern=False,
                         dropout=0.2,
                         batch_norm=True,
                         regularizer=L2(l2=1e-5)
                        )

#Compiling the model
unet_model.compile(loss=dice_loss,
                   optimizer=Adam(learning_rate=lr_init),
                   metrics=[dice_coefficient,
                            OneHotIoU(num_classes=num_classes, target_class_ids=[i for i in range(num_classes)]),
                            CategoricalAccuracy()])

#Loading the most recent saved model weights to continue the training
if os.path.exists(checkpoint_filepath_unet):
    print("Loading...")
    unet_model.load_weights(checkpoint_filepath_unet)

#Training the model
history = unet_model.fit(train_dataset_augment,
                         validation_data=val_dataset_augment,
                         epochs=epochs,
                         callbacks=[
                             early_stop,
                             ckpt_callback_unet,
                             clr,
                             csvlogger_unet,
                             DisplayCallback_baseline()
                         ],
                         batch_size=batch_size,
                         validation_batch_size=batch_size,
                         verbose=1,
                         class_weight=dict(zip([0, 1, 2, 3], classes_weight))
                        )

"""unet_model.compile(loss=(#tf.keras.losses.CategoricalFocalCrossentropy(alpha=classes_weight),
                         #"categorical_crossentropy",
                         dice_loss
                         #WLCE,
                        ),
                   optimizer=Adam(learning_rate=lr_init),
                   metrics=[tf.keras.metrics.OneHotIoU(num_classes=num_classes, target_class_ids=[i for i in range(num_classes)]),
                            tf.keras.metrics.CategoricalAccuracy(),
                            dice_coefficient,
                           ]
                  )"""

### U-NET with ResNet34 as encoder and ERN as reinforcement

import shutil

shutil.rmtree(checkpoint_filepath_unet_ern)
os.remove(f"history{version_unet_ern}.csv")

In [ ]:
#Create a instance of my Unet class
epochs = 100
unet_model = MyUnetModel(num_classes=num_classes,
                         size_image=size_img,
                         backbone=True,
                         training_up_to_layer=37,
                         ern=True,
                         dropout=0.2,
                         batch_norm=True,
                         regularizer=L2(l2=1e-5)
                        )

#Compiling the model
unet_model.compile(loss={
                         "segmentation": dice_loss,
                         "edge_detection_encoder": BinaryFocalCrossentropy(apply_class_balancing=True, alpha=1-weight_edge_pixel),
                         "edge_detection_decoder": BinaryFocalCrossentropy(apply_class_balancing=True, alpha=1-weight_edge_pixel),
                        },
                   optimizer=Adam(learning_rate=lr_init),
                   metrics={
                       "segmentation": [dice_coefficient],
                       "edge_detection_encoder": [BinaryCrossentropy()],
                       "edge_detection_decoder": [BinaryCrossentropy()],
                       }
                  )

#Loading the most recent saved model weights to continue the training
if os.path.exists(checkpoint_filepath_unet_ern):
    print("Loading...")
    unet_model.load_weights(checkpoint_filepath_unet_ern)

#Training the model
history = unet_model.fit(train_dataset_edge,
                         validation_data=val_dataset_edge,
                         epochs=epochs,
                         callbacks=[
                             early_stop,
                             ckpt_callback_unet_ern,
                             clr,
                             csvlogger_unet_ern,
                             DisplayCallback()
                         ],
                         batch_size=batch_size,
                         validation_batch_size=batch_size,
                         verbose=1,
                         #class_weight=dict(zip([0, 1, 2, 3], classes_weight))
                        )

## Showing the changes in the parameters over epochs during training.

In [ ]:
def show_history(path_csv, parameters=None):
    df_hist = pd.read_csv(path_csv)
    fig, axs = plt.subplots(2, 4, figsize=(22, 8))

    for i, col in enumerate(parameters):
        plt.subplot(2, 4, i+1)
        param = col[0].replace('_', ' ').title()
        plt.plot(df_hist.index, df_hist[col[0]], label=f"Training {param}")
        plt.plot(df_hist.index, df_hist[col[1]], label=f"Validation {param}")
        plt.ylabel(col[0].capitalize().replace("_", " "))
        plt.xlabel("Epoch")
        plt.legend(fontsize=8)
        plt.grid(True, alpha=0.3, which="both")
        plt.title(f"{param} over epochs")

    fig, axs = delete_empty_subplots(fig, axs)
    plt.tight_layout()
    plt.show()
    return

version = "ern"
show_history(f"/kaggle/working/history{version}.csv", [["loss", "val_loss"],
                                                       ["segmentation_loss", "val_segmentation_loss"],
                                                       ["edge_detection_encoder_loss", "val_edge_detection_encoder_loss"],
                                                       ["edge_detection_decoder_loss", "val_edge_detection_decoder_loss"],
                                                       ["segmentation_dice_coefficient", "val_segmentation_dice_coefficient"],
                                                       ["edge_detection_encoder_binary_crossentropy", "val_edge_detection_encoder_binary_crossentropy"],
                                                       ["edge_detection_decoder_binary_crossentropy", "val_edge_detection_decoder_binary_crossentropy"]
                                                       #"categorical_accuracy",
                                                       #"one_hot_io_u",
                                                      ])

# Reloading the models weights

In [ ]:
### Baseline UNet
unet_model_baseline = MyUnetModel(num_classes=num_classes,
                         size_image=size_img,
                         backbone=False,
                         ern=False,
                         dropout=0.2,
                         batch_norm=True,
                         regularizer=L2(l2=1e-5)
                        )


unet_model_baseline.load_weights(checkpoint_filepath_baseline)

unet_model_baseline.compile(loss=dice_loss,
                   optimizer=Adam(learning_rate=lr_init),
                   metrics=[dice_coefficient,
                            OneHotIoU(num_classes=num_classes, target_class_ids=[i for i in range(num_classes)]),
                            CategoricalAccuracy()])

###UNet+Resnet34
max_filters = 256
unet_model_resnet = MyUnetModel(num_classes=num_classes,
                         size_image=size_img,
                         backbone=True,
                         training_up_to_layer=37,
                         mini_blocks=3,
                         ern=False,
                         dropout=0.2,
                         batch_norm=True,
                         regularizer=L2(l2=1e-5)
                        )

unet_model_resnet.load_weights(checkpoint_filepath_unet)

unet_model_resnet.compile(loss=dice_loss,
                   optimizer=Adam(learning_rate=lr_init),
                   metrics=[dice_coefficient,
                            OneHotIoU(num_classes=num_classes, target_class_ids=[i for i in range(num_classes)]),
                            CategoricalAccuracy()])

### UNet+Resnet34+ERN
version = "ern"

checkpoint_filepath_unet_ern = dst_path + f'modelckpt{version}/checkpoint_basic_unet.ckpt'

max_filters = 256
unet_model = MyUnetModel(num_classes=num_classes,
                         size_image=size_img,
                         backbone=True,
                         training_up_to_layer=37,
                         ern=True,
                         dropout=0.2,
                         batch_norm=True,
                         regularizer=L2(l2=1e-5)
                        )

unet_model.load_weights(checkpoint_filepath_unet_ern)

unet_model.compile(loss={
                         "segmentation": dice_loss,
                         "edge_detection_encoder": BinaryFocalCrossentropy(apply_class_balancing=True, alpha=1-weight_edge_pixel),
                         "edge_detection_decoder": BinaryFocalCrossentropy(apply_class_balancing=True, alpha=1-weight_edge_pixel),
                        },
                   metrics={
                       "segmentation": [dice_coefficient,
                                        OneHotIoU(num_classes=num_classes, target_class_ids=[i for i in range(num_classes)]),
                                        CategoricalAccuracy()],
                       "edge_detection_encoder": [tf.keras.metrics.BinaryCrossentropy()],
                       "edge_detection_decoder": [tf.keras.metrics.BinaryCrossentropy()],
                       }
                  )

# Evaluation metrics

In [ ]:
val_metrics_unet_baseline = unet_model_baseline.evaluate(val_dataset_augment, batch_size=batch_size)
test_metrics_unet_baseline = unet_model_baseline.evaluate(test_dataset_augment, batch_size=batch_size)

val_metrics_unet_resnet = unet_model_resnet.evaluate(val_dataset_augment, batch_size=batch_size)
test_metrics_unet_resnet = unet_model_resnet.evaluate(test_dataset_augment, batch_size=batch_size)

val_metrics_unet_ern = unet_model.evaluate(val_dataset_edge, batch_size=batch_size)
test_metrics_unet_ern = unet_model.evaluate(test_dataset_edge, batch_size=batch_size)

## Comparison table

In [ ]:
columns = ["Segmentation Loss",
           "Dice Coefficient",
           "One-Hot IOU",
           "Categorical Accuracy"
          ]

test_scores = pd.DataFrame(data=np.stack([test_metrics_unet_baseline, test_metrics_unet_resnet, [test_metrics_unet_ern[3]] + test_metrics_unet_ern[6:]]), columns=columns, index = ["U-Net Baseline", "U-Net + Resnet34", "U-Net + Resnet34 + ERN"]).T

test_scores.style.highlight_max(color = 'green', axis = 1).highlight_min(color = 'red', axis = 1)

In [ ]:
columns = ["Segmentation Loss",
           "Dice Coefficient",
           "One-Hot IOU",
           "Categorical Accuracy"]

val_scores = pd.DataFrame(data=np.stack([val_metrics_unet_baseline, val_metrics_unet_resnet, [val_metrics_unet_ern[3]] + val_metrics_unet_ern[6:]]), columns=columns, index = ["U-Net Baseline", "U-Net + Resnet34", "U-Net + Resnet34 + ERN"]).T

val_scores.style.highlight_max(color = 'green', axis = 1).highlight_min(color = 'red', axis = 1)

The metrics are similars in both datasets, our model isn't overfit either underfit.

The Dice coefficient is 0.86, this is a very good value. A high Dice coefficient value indicates a high level of similarity between the predicted and ground truth masks, meaning that the segmentation model or algorithm is performing well. Conversely, a low Dice coefficient value indicates poor segmentation performance.

## Confusion Matrix

### Mask prediction on the testing set

In [ ]:
y_pred_test = unet_model_baseline.predict(test_dataset_augment)
pred_mask  = tf.math.argmax(y_pred_test, axis=-1).numpy().flatten().astype('uint8')

### Truth mask on the testing set

In [ ]:
mask = []
for image, output in test_dataset_augment.take(np.ceil(test_size/batch_size).astype(int)).unbatch():
    mask.append(output)
mask = np.argmax(np.array(mask), axis=-1).flatten().astype('uint8')

### Calculating the confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

def my_cm(y_true, y_pred, title):
    cm_val = confusion_matrix(y_true, y_pred)
    cm_pgs = np.round(confusion_matrix(y_true, y_pred, normalize='true')*100, 4)

    formatted_text = (np.asarray([f"{pgs}%\n({val})" for val, pgs in zip(cm_val.flatten(), cm_pgs.flatten())])).reshape(4, 4)

    sns.heatmap(cm_pgs, xticklabels=Labels.get_classes(), yticklabels=Labels.get_classes(), annot=formatted_text, fmt='', cmap='BuPu')
    plt.title(title)
    plt.xlabel("Prediction")
    plt.ylabel("Expected")

    plt.subplots_adjust(hspace=0.5)
    return

plt.figure(figsize=(8, 8))
my_cm(mask, pred_mask, title="Confusion matrix by pixel classification")

# Showing some predictions

## Example 1

In [ ]:
def predict_mask(image=None, mask=None, df_colors=None, model=None, plot=True, normalize=True):
    color_rgb = tf.constant(df_colors["Color_RGB"].to_list(), dtype=tf.int32)
    color_conv = tf.constant(df_colors["Conversion"].to_list(), dtype=tf.int32)

    size_img = 256
    if normalize:
        image = normalize_size(image, 3*size_img)
    crops_image = crop_image(image, size_img)

    pred_mask = model.predict(tf.cast(crops_image, tf.float32)/255)#["segmentation"]
    pred_crops_mask = []
    for crop_mask in pred_mask:
        pred_crops_mask.append(ohe2rgb(crop_mask, Labels.get_colors()))

    pred_crops_mask = tf.convert_to_tensor(pred_crops_mask, tf.uint8)

    img_shape = tf.shape(image)
    restored_mask = restore_image(pred_crops_mask, img_shape)

    if mask is not None:
        mask_cor = mask_color_correction(mask, color_rgb)
        mask_norm = norm_colors(mask_cor, color_rgb, color_conv)
        mask = normalize_size(mask_norm, 3*size_img)
        n_subplots = 3
        if plot == True:
            plt.subplots(1, n_subplots, figsize=(15, 5))
    else:
        n_subplots = 2
        if plot == True:
            plt.subplots(1, n_subplots, figsize=(10, 5))

    i = 1
    if plot == True:
        plt.subplot(1, n_subplots, i)
        plt.imshow(image)
        plt.title("Original image", fontsize=9)
        plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
    i += 1

    if mask is not None:
        if plot == True:
            plt.subplot(1, n_subplots, i)
            plt.imshow(mask)
            plt.title("Original mask", fontsize=9)
            plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
        i += 1


    if plot == True:
        plt.subplot(1, n_subplots, i)
        plt.imshow(restored_mask)
        plt.title("Predicted mask", fontsize=9)
        plt.tick_params(bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
        plt.tight_layout()
    return image, restored_mask

img = load_img("/kaggle/input/urban-segmentation-isprs/Potsdam/Images/top_potsdam_6_8_RGB.tif")
msk = load_img("/kaggle/input/urban-segmentation-isprs/Potsdam/Labels/top_potsdam_6_8_label.tif")[..., ::-1]

image, restored_mask = predict_mask(img, msk, ISPRS_colors, model=unet_model_baseline, plot=True)

## Example 2

In [ ]:
pathfile_test = [
                "buenos_aires_33.tif",
                #"chiangmai_13.tif",
                #"chicago_33.tif",
                #"chicago_47.tif"
                ]
img = load_img(f"/kaggle/input/global-land-cover-mapping-openearthmap/images/test/{pathfile_test[0]}")

image, restored_mask = predict_mask(img, df_colors=ISPRS_colors, model=unet_model_baseline, plot=True)

## Decompositing the mask prediction

In [ ]:
mask_ohe = rgb2ohe(restored_mask, Labels.get_colors())
plot_decomposition(tf.cast(image, dtype=tf.uint8), restored_mask, mask_ohe, SSAI_colors, num_classes=num_classes)

### Cropping images with the masks

In [ ]:
def plot_crop_image_with_mask(image, mask=None, mask_ohe=None, df_colors=None, figsize=(15, 25), rows=3, cols=2):

    num_classes = tf.shape(mask_ohe)[-1].numpy()
    fig, axs = plt.subplots(rows, cols, figsize=figsize)

    if mask is not None:
        plot_sample(image, mask, df_colors=SSAI_colors, num_img=rows, num_cols=cols)
        ind = 3
    else:
        subplot(image=image, num_row=rows, num_cols=cols, idx=1, title="Aerial image")
        ind = 2

    labels = Labels.get_classes()
    for i in range(num_classes):
        image_cut = tf.cast(tf.expand_dims(mask_ohe[:, :, i], axis=-1), tf.uint8) * image
        subplot(image=image_cut, num_row=rows, num_cols=cols, idx=ind, title=labels[i] + " mask")
        ind += 1
    plt.tight_layout()
    plt.show()
    return

plot_crop_image_with_mask(image, restored_mask, mask_ohe, SSAI_colors, figsize=(20, 30), rows=3, cols=2)

In [ ]:
pathfile_test = [
                #"buenos_aires_33.tif",
                #"chiangmai_13.tif",
                #"chicago_33.tif",
                #"chicago_47.tif",
                #"kyoto_30.tif",
                "bogota_7.tif"
                ]
img = load_img("/kaggle/input/swiss-drone-and-okutama-drone-datasets/images/test/okutama_04_90_016.png")
image, restored_mask = predict_mask(img, df_colors=ISPRS_colors, model=unet_model_baseline, plot=False)
mask_ohe = rgb2ohe(restored_mask, Labels.get_colors())
plot_crop_image_with_mask(image, restored_mask, mask_ohe, SSAI_colors, figsize=(20, 18), rows=3, cols=2)

# Extra: Downloading aerial images from [Google Earth Pro](https://www.google.com/intl/es/earth/about/versions/)



The following aerial images were donwloaded by following this [intructions](https://support.google.com/earth/answer/148146?hl=en). We'll segment these images with our model.

In [ ]:
if not os.path.exists("/kaggle/working/MyAerialImages/"):
    os.mkdir("/kaggle/working/MyAerialImages/")

id_list = ["1fyNzITkOcr2cs4LRf2CRMkIVhrtYXxWM",
           "15R-S05m7GznYuJleusXd7VI4XADshzuo",
           "1JXdTj6ShBChSIw1ykZE4Xu3E03EnhXdn",
           "123buOGE5XVvhvfIcvpZpx7ce_UoLXoKj"]

for n, idx in enumerate(id_list):
    url = f'https://drive.google.com/uc?id={idx}'
    output = f'/kaggle/working/MyAerialImages/{n+1}.jpg'
    gdown.download(url, output)

## The UTN headquarters in Medrano and its surroundings

In [ ]:
pathfile_test = ["/kaggle/working/MyAerialImages/1.jpg",
                 "/kaggle/working/MyAerialImages/2.jpg",
                 "/kaggle/working/MyAerialImages/3.jpg",
                 "/kaggle/working/MyAerialImages/4.jpg"]

img = load_img(pathfile_test[0])
res_img = resize_image(img=img, major_size=size_img*10, padding=False)
image, restored_mask = predict_mask(res_img, df_colors=ISPRS_colors, model=unet_model_baseline, plot=False, normalize=False)
mask_ohe = rgb2ohe(restored_mask, Labels.get_colors())
plot_crop_image_with_mask(image, restored_mask, mask_ohe, SSAI_colors, figsize=(40, 30), rows=6, cols=1)

## The UTN headquarters on Campus and its surroundings

In [ ]:
img = load_img(pathfile_test[2])
res_img = resize_image(img=img, major_size=size_img*5, padding=False)
image, restored_mask = predict_mask(res_img, df_colors=ISPRS_colors, model=unet_model_baseline, plot=False, normalize=False)
mask_ohe = rgb2ohe(restored_mask, Labels.get_colors())
plot_crop_image_with_mask(image, restored_mask, mask_ohe, SSAI_colors, figsize=(40, 30), rows=6, cols=1)

## My neighborhood

In [ ]:
img = load_img(pathfile_test[1])
res_img = resize_image(img=img, major_size=size_img*15, padding=False)
image, restored_mask = predict_mask(res_img, df_colors=ISPRS_colors, model=unet_model_baseline, plot=False, normalize=False)
mask_ohe = rgb2ohe(restored_mask, Labels.get_colors())
plot_crop_image_with_mask(image, restored_mask, mask_ohe, SSAI_colors, figsize=(40, 30), rows=6, cols=1)

## Random location

In [ ]:
img = load_img(pathfile_test[3])
res_img = resize_image(img=img, major_size=size_img*10, padding=False)
image, restored_mask = predict_mask(res_img, df_colors=ISPRS_colors, model=unet_model_baseline, plot=False, normalize=False)
mask_ohe = rgb2ohe(restored_mask, Labels.get_colors())
plot_crop_image_with_mask(image, restored_mask, mask_ohe, SSAI_colors, figsize=(40, 30), rows=6, cols=1)

# Converting Notebook to PDF

In [ ]:
!apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic pandoc
!pip install pypandoc nbconvert[webpdf]
!playwright install

from google.colab import drive
from IPython.display import clear_output
drive.mount('/content/drive')
clear_output(wait=False)

In [ ]:
%%capture
!jupyter nbconvert --to webpdf /content/drive/MyDrive/Proyecto_Final_-_Aprendizaje_Automatico_-_Durante/aerial-image-for-semantic-segmentation_final.ipynb --HTMLExporter.theme=dark